In [ ]:
# Cell 1 — Objective: Install only the required packages for this notebook; import nothing sensitive here.

%pip install --quiet --upgrade pip
%pip install --quiet oci ipywidgets ipyfilechooser paramiko tqdm numpy pandas locust requests

print("Cell 1 complete: Python environment ready.")
print("NEXT: Run Cell 2 to load central variables and validate OCI config (LB offload TLS + PPv2; TCP/80 health).")


In [ ]:
# Cell 2 — Objective: Central variables and credentials (single source of truth; no network calls)
# Adds certificate generation controls and a lazy generator function (not executed here).
# Also adds deterministic PEM file naming (LB_PEM_BASENAME) to avoid duplicates.

import os
from datetime import datetime, timezone
import oci

# OCI config (no secrets hardcoded; load from standard ~/.oci/config)
OCI_CONFIG_FILE = os.path.expanduser(os.environ.get("OCI_CONFIG_FILE", "~/.oci/config"))
OCI_PROFILE = os.environ.get("OCI_PROFILE", "DEFAULT")

_cfg = oci.config.from_file(file_location=OCI_CONFIG_FILE, profile_name=OCI_PROFILE)
oci.config.validate_config(_cfg)

TENANCY_OCID = os.environ.get("TENANCY_OCID", _cfg["tenancy"])
USER_OCID = os.environ.get("USER_OCID", _cfg["user"])
FINGERPRINT = os.environ.get("FINGERPRINT", _cfg["fingerprint"])
OCI_PRIVATE_KEY_PATH = os.path.expanduser(
    os.environ.get("OCI_PRIVATE_KEY_PATH", _cfg.get("key_file", ""))
)
PRIVATE_KEY_PASSPHRASE = os.environ.get(
    "OCI_PASSPHRASE",
    os.environ.get("OCI_PRIVATE_KEY_PASSPHRASE", _cfg.get("pass_phrase", "")),
)
REGION = os.environ.get("REGION", _cfg["region"])

# SSH keys defaults (can be changed in Cell 3)
_default_ssh_pub = os.path.expanduser(
    os.environ.get("SSH_PUBLIC_KEY_PATH", "~/.ssh/id_rsa.pub")
)
_default_ssh_priv = os.path.expanduser(
    os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/id_rsa")
)
SSH_PUBLIC_KEY_PATH = _default_ssh_pub if os.path.exists(_default_ssh_pub) else ""
SSH_PRIVATE_KEY_PATH = _default_ssh_priv if os.path.exists(_default_ssh_priv) else ""

# TLS OFFLOAD: Either browse existing PEMs or generate self‑signed in Cell 3
# Mode: "browse" (default) or "generate"
LB_CERT_MODE = (
    os.environ.get("LB_CERT_MODE", "browse").strip().lower()
)  # "browse" | "generate"

# If browsing existing PEMs (these will be set via Cell 3 pickers or environment)
LB_CERT_PEM_PATH = os.path.expanduser(os.environ.get("LB_CERT_PEM_PATH", ""))
LB_KEY_PEM_PATH = os.path.expanduser(os.environ.get("LB_KEY_PEM_PATH", ""))
LB_CA_PEM_PATH = os.path.expanduser(os.environ.get("LB_CA_PEM_PATH", ""))  # optional

# If generating PEMs (defaults suitable for high‑CPS)
LB_CERT_KIND = (
    os.path.expanduser(os.environ.get("LB_CERT_KIND", "ecdsa")).strip().lower()
)  # "ecdsa" | "rsa"
LB_ECDSA_CURVE = os.environ.get("LB_ECDSA_CURVE", "secp256r1").strip()  # P‑256
LB_RSA_BITS = int(os.environ.get("LB_RSA_BITS", "2048"))
LB_CERT_CN = os.environ.get("LB_CERT_CN", "lb.local").strip()
LB_CERT_DAYS = int(os.environ.get("LB_CERT_DAYS", "3650"))
LB_PEM_OUTPUT_DIR = os.path.expanduser(
    os.environ.get("LB_PEM_OUTPUT_DIR", "./local-lb-pems")
)

# Deterministic PEM basename so Generate always writes the same filenames
LB_PEM_BASENAME = os.environ.get("LB_PEM_BASENAME", "lb_current").strip()

# Topology defaults (adjustable in Cell 3)
BACKEND_COUNT = int(os.environ.get("BACKEND_COUNT", "8"))
GENERATOR_COUNT = int(os.environ.get("GENERATOR_COUNT", "8"))

BACKEND_SHAPE = os.environ.get("BACKEND_SHAPE", "VM.Standard.E5.Flex")
BACKEND_OCPUS = float(os.environ.get("BACKEND_OCPUS", "16"))
BACKEND_MEMORY_GB = float(os.environ.get("BACKEND_MEMORY_GB", "64"))

GENERATOR_SHAPE = os.environ.get("GENERATOR_SHAPE", "VM.Standard.E5.Flex")
GENERATOR_OCPUS = float(os.environ.get("GENERATOR_OCPUS", "16"))
GENERATOR_MEMORY_GB = float(os.environ.get("GENERATOR_MEMORY_GB", "64"))

# Single‑LB bandwidth caps (Mbps) — used by tfvars in Cell 6
LB_MIN_MBPS = int(os.environ.get("LB_MIN_MBPS", "8000"))
LB_MAX_MBPS = int(os.environ.get("LB_MAX_MBPS", "8000"))

# Endpoints and Locust behavior
HEALTH_ENDPOINT_PATH = os.environ.get("HEALTH_ENDPOINT_PATH", "/healthz")
THROUGHPUT_ENDPOINT_PATH = os.environ.get("THROUGHPUT_ENDPOINT_PATH", "/payload_100k")
LOCUST_WAIT_TIME_SEC = float(os.environ.get("LOCUST_WAIT_TIME_SEC", "1.0"))
LOCUST_CONNECT_TIMEOUT = int(os.environ.get("LOCUST_CONNECT_TIMEOUT_MS", "8000"))  # ms
LOCUST_READ_TIMEOUT = int(os.environ.get("LOCUST_READ_TIMEOUT_MS", "15000"))  # ms
LOCUST_VERIFY_TLS = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"

# Remote workspace (must NOT be named "locust" to avoid Python package shadowing)
LOCUST_WORKDIR = os.environ.get("LOCUST_WORKDIR", "/home/opc/locustwork")

# Worker policy
EXPECT_WORKERS_STRICT = (
    os.environ.get("EXPECT_WORKERS_STRICT", "false").lower() == "true"
)
EXPECTED_WORKERS_OVERRIDE = (
    None
    if os.environ.get("EXPECTED_WORKERS_OVERRIDE", "").strip() == ""
    else int(os.environ.get("EXPECTED_WORKERS_OVERRIDE"))
)
GRACE_SEC = int(os.environ.get("GRACE_SEC", "60"))

# UI
UI_ENABLE = True
UI_EXPOSE_MODE = "tunnel"  # "tunnel" or "nsg"
UI_WEB_HOST = "0.0.0.0"
UI_WEB_PORT = int(os.environ.get("UI_WEB_PORT", "8089"))
UI_ALLOWED_CIDR = os.environ.get("UI_ALLOWED_CIDR", "0.0.0.0/0")

# Test Mode selector
TEST_MODE = "cps"  # "cps" or "throughput"

# CPS tiers & durations (overridable in Cell 3)
CPS_TIERS = [10000, 25000, 35000, 50000, 100000]
CPS_WARMUPS = {10000: 120, 25000: 150, 35000: 160, 50000: 180, 100000: 210}
CPS_HOLDS = {10000: 600, 25000: 600, 35000: 600, 50000: 600, 100000: 600}

# Throughput configuration defaults
TPUT_TARGETS_GBPS_TEXT = "1,5,10"
TPUT_WARMUP_SEC = 120
TPUT_HOLD_SEC = 600

# Payload knobs (will be finalized in Cell 3)
TPUT_PAYLOAD_SIZE_TEXT = "100k"
TPUT_PAYLOAD_SIZES_TEXT = "4k,5k,10k,50k,100k,256k,1m,5m"

# Derived (set in Cell 3)
TPUT_PAYLOAD_SIZE_BYTES = 100_000
TPUT_PAYLOAD_SIZE_LABEL = "100k"
TPUT_PAYLOAD_BYTES_PER_REQ = TPUT_PAYLOAD_SIZE_BYTES

# Generator CPU → worker policy
WORKERS_PER_HOST = "auto"
CPU_RESERVE = 1
MIN_WORKERS_PER_HOST = 1
MAX_WORKERS_PER_HOST = 32

MASTER_PRIVATE_IP_OVERRIDE = os.environ.get("MASTER_PRIVATE_IP_OVERRIDE", "").strip()
SSH_ALLOWED_CIDR = os.environ.get("SSH_ALLOWED_CIDR", "0.0.0.0/0")

OUTPUT_DIR = os.path.abspath(os.environ.get("OUTPUT_DIR", "./results"))
os.makedirs(OUTPUT_DIR, exist_ok=True)
TS_UTC = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

# Placeholders (populated in Cell 3)
SELECTED_REGION = None
COMPARTMENT_ID = ""
AD_A = ""
AD_B = ""  # kept for UI parity; Single‑LB uses one subnet (AD_A)
IMAGE_ID = ""
SSH_PUBLIC_KEY_CONTENT = ""


def _ensure_path_exists(path: str, label: str, optional=False):
    if not path:
        if optional:
            return
        raise FileNotFoundError(f"{label} path not set.")
    p = os.path.expanduser(path)
    if not os.path.exists(p):
        if optional:
            return
        raise FileNotFoundError(f"{label} not found: {p}")


_ensure_path_exists(OCI_PRIVATE_KEY_PATH, "OCI private key", optional=False)


# Lazy certificate generator (NOT executed here; used by Cell 3 when LB_CERT_MODE == "generate")
def generate_self_signed_lb_pems(
    kind: str, curve_name: str, rsa_bits: int, cn: str, days: int, out_dir: str
):
    """
    Generate self-signed PEMs for LB TLS offload.
    Returns (cert_path, key_path, ca_path).
    Uses deterministic filenames via LB_PEM_BASENAME (overwrites on each generate).
    """
    # Local import to avoid adding deps unless used
    try:
        from cryptography import x509
        from cryptography.x509.oid import NameOID
        from cryptography.hazmat.primitives import hashes, serialization
        from cryptography.hazmat.primitives.asymmetric import ec, rsa
        from cryptography.hazmat.backends import default_backend
        from datetime import datetime as dt2, timedelta
    except ImportError:
        import sys, subprocess

        print("Installing cryptography ...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "cryptography"]
        )
        from cryptography import x509
        from cryptography.x509.oid import NameOID
        from cryptography.hazmat.primitives import hashes, serialization
        from cryptography.hazmat.primitives.asymmetric import ec, rsa
        from cryptography.hazmat.backends import default_backend
        from datetime import datetime as dt2, timedelta

    kind = (kind or "ecdsa").strip().lower()
    out_dir = os.path.expanduser(out_dir or "./local-lb-pems")
    os.makedirs(out_dir, exist_ok=True)
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

    if kind == "ecdsa":
        curve_l = (curve_name or "secp256r1").strip().lower()
        curve_map = {
            "secp256r1": ec.SECP256R1(),
            "prime256v1": ec.SECP256R1(),
            "p-256": ec.SECP256R1(),
        }
        curve = curve_map.get(curve_l, ec.SECP256R1())
        key = ec.generate_private_key(curve, backend=default_backend())
        kind_tag = "ecdsa"
    else:
        key = rsa.generate_private_key(
            public_exponent=65537,
            key_size=int(rsa_bits or 2048),
            backend=default_backend(),
        )
        kind_tag = f"rsa{int(rsa_bits or 2048)}"

    subject = issuer = x509.Name(
        [x509.NameAttribute(NameOID.COMMON_NAME, cn or "lb.local")]
    )
    builder = (
        x509.CertificateBuilder()
        .subject_name(subject)
        .issuer_name(issuer)
        .public_key(key.public_key())
        .serial_number(x509.random_serial_number())
        .not_valid_before(dt2.utcnow())
        .not_valid_after(dt2.utcnow() + timedelta(days=int(days or 3650)))
        .add_extension(x509.BasicConstraints(ca=False, path_length=None), critical=True)
    )
    try:
        builder = builder.add_extension(
            x509.SubjectAlternativeName([x509.DNSName(cn or "lb.local")]),
            critical=False,
        )
    except Exception:
        pass

    cert = builder.sign(
        private_key=key, algorithm=hashes.SHA256(), backend=default_backend()
    )

    key_pem = key.private_bytes(
        encoding=serialization.Encoding.PEM,
        format=serialization.PrivateFormat.PKCS8,
        encryption_algorithm=serialization.NoEncryption(),
    )
    cert_pem = cert.public_bytes(serialization.Encoding.PEM)

    # Deterministic base name: prefer LB_PEM_BASENAME, else fall back to timestamped
    base = LB_PEM_BASENAME if LB_PEM_BASENAME else f"lb_{kind_tag}_{ts}"
    key_path = os.path.join(out_dir, f"{base}.key.pem")
    cert_path = os.path.join(out_dir, f"{base}.cert.pem")
    ca_path = os.path.join(out_dir, f"{base}.ca.pem")  # empty for self-signed

    with open(key_path, "wb") as f:
        f.write(key_pem)
    with open(cert_path, "wb") as f:
        f.write(cert_pem)
    with open(ca_path, "wb") as f:
        f.write(b"")

    return cert_path, key_path, ca_path


if not SSH_PUBLIC_KEY_PATH:
    print(
        "Note: SSH public key not found at default path; you will select a key in Cell 3."
    )
if not SSH_PRIVATE_KEY_PATH:
    print(
        "Note: SSH private key not found at default path; you will select a key in Cell 3."
    )

print(
    f"Cell 2 complete: Config loaded from {OCI_CONFIG_FILE} [{OCI_PROFILE}] for tenancy: {TENANCY_OCID}"
)
print(
    "NEXT: Run Cell 3, choose Cert Mode = Browse or Generate (and parameters), set shapes/counts, then Apply."
)

In [ ]:
# Cell 3 — Objective: Interactive dashboard with Cert Mode (Browse or Generate), TLS PEM handling, and Apply.
# Deterministic PEM support + safe generate: only overwrite if requested.

import os
from pathlib import Path
import oci
from IPython.display import display, HTML
import ipywidgets as widgets
from ipyfilechooser import FileChooser

SECTION_BG = "#f8f9fb"
BORDER = "1px solid #e0e0e0"
PAD = "12px"
CONTAINER_MAX_W = "96%"


def section(title_text, body_widgets):
    title = widgets.HTML(f"<b>{title_text}</b>")
    body = (
        body_widgets
        if isinstance(body_widgets, widgets.Widget)
        else widgets.VBox(body_widgets)
    )
    return widgets.VBox(
        [title, body],
        layout=widgets.Layout(
            width="100%",
            border=BORDER,
            padding=PAD,
            margin="8px 0",
            background_color=SECTION_BG,
        ),
    )


def row(*children, gap="12px"):
    return widgets.HBox(list(children), layout=widgets.Layout(gap=gap, width="100%"))


def vspace(h="6px"):
    return widgets.HTML(f"<div style='height:{h}'></div>")


def build_signer(tenancy, user, fp, key_path, passphrase, cfg):
    return oci.signer.Signer(
        tenancy=tenancy,
        user=user,
        fingerprint=fp,
        private_key_file_location=key_path,
        pass_phrase=passphrase if passphrase else None,
        private_key_content=cfg.get("key_content"),
    )


def cfg_for_region(region_name):
    c = dict(_cfg)
    c["region"] = region_name
    return c


# Region selector
_base_signer = build_signer(
    TENANCY_OCID,
    USER_OCID,
    FINGERPRINT,
    OCI_PRIVATE_KEY_PATH,
    PRIVATE_KEY_PASSPHRASE,
    _cfg,
)
idc_base = oci.identity.IdentityClient(config=_cfg, signer=_base_signer)
subs = sorted(
    idc_base.list_region_subscriptions(TENANCY_OCID).data, key=lambda r: r.region_name
)
region_options = [(r.region_name, r.region_name) for r in subs]
default_region = (
    REGION
    if REGION in [r.region_name for r in subs]
    else (next((r.region_name for r in subs if r.is_home_region), subs[0].region_name))
)
region_dd = widgets.Dropdown(
    options=region_options,
    value=default_region,
    description="Region:",
    layout=widgets.Layout(width="100%"),
)
reload_btn = widgets.Button(description="Reload", icon="refresh")
reset_btn = widgets.Button(description="Reset", icon="history")
err_out, summary_out = widgets.Output(), widgets.Output()
dynamic_box = widgets.VBox([])

# SSH key dropdowns


def discover_files(dirs, exts=None, include_hidden=True):
    out = []
    for d in dirs:
        p = Path(os.path.expanduser(d))
        if not p.exists() or not p.is_dir():
            continue
        for f in p.iterdir():
            if not f.is_file():
                continue
            if not include_hidden and f.name.startswith("."):
                continue
            if exts is not None and not any(str(f).endswith(ext) for ext in exts):
                continue
            out.append(str(f))
    out.sort(
        key=lambda s: Path(s).stat().st_mtime if Path(s).exists() else 0, reverse=True
    )
    return out


ssh_dir = os.path.expanduser("~/.ssh")
ssh_pub_candidates = [p for p in discover_files([ssh_dir], exts=[".pub"])]
ssh_priv_candidates = [
    p for p in discover_files([ssh_dir], exts=None) if not p.endswith(".pub")
]
ssh_pub_default = next(
    (p for p in ssh_pub_candidates if p == SSH_PUBLIC_KEY_PATH),
    (ssh_pub_candidates[0] if ssh_pub_candidates else ""),
)
ssh_priv_default = next(
    (p for p in ssh_priv_candidates if p == SSH_PRIVATE_KEY_PATH),
    (ssh_priv_candidates[0] if ssh_priv_candidates else ""),
)

ssh_pub_dd = widgets.Dropdown(
    options=[(p, p) for p in ssh_pub_candidates]
    or [("No *.pub keys found in ~/.ssh", "")],
    value=ssh_pub_default,
    description="SSH pub:",
    layout=widgets.Layout(width="100%"),
)
ssh_priv_dd = widgets.Dropdown(
    options=[(p, p) for p in ssh_priv_candidates]
    or [("No private keys found in ~/.ssh", "")],
    value=ssh_priv_default,
    description="SSH priv:",
    layout=widgets.Layout(width="100%"),
)

# Cert Mode selector
cert_mode_dd = widgets.Dropdown(
    options=[("Browse PEM files", "browse"), ("Generate self-signed PEMs", "generate")],
    value=LB_CERT_MODE,
    description="Cert Mode:",
)

# Browse widgets
home = os.path.expanduser("~")
fc_cert = FileChooser(
    path=home,
    title="Select LB certificate (PEM/CRT/CER)",
    show_hidden=True,
    use_dir_icons=True,
    show_only_dirs=False,
    filter_pattern="*",
)
fc_key = FileChooser(
    path=home,
    title="Select LB private key (.key or .pem)",
    show_hidden=True,
    use_dir_icons=True,
    show_only_dirs=False,
    filter_pattern="*",
)
fc_ca = FileChooser(
    path=home,
    title="Optional: Select CA/chain PEM",
    show_hidden=True,
    use_dir_icons=True,
    show_only_dirs=False,
    filter_pattern="*",
)
show_hidden_cb = widgets.Checkbox(
    value=True, description="Show hidden files (dot folders)"
)
jump_path_tb = widgets.Text(
    value=os.path.expanduser("~"),
    description="Jump to folder:",
    layout=widgets.Layout(width="100%"),
)
jump_btn = widgets.Button(description="Go", icon="sign-in")


def _set_show_hidden(val: bool):
    for fc in (fc_cert, fc_key, fc_ca):
        try:
            fc.show_hidden = val
        except Exception:
            pass


def _jump_all(path: str):
    p = os.path.expanduser(path or "")
    for fc in (fc_cert, fc_key, fc_ca):
        try:
            fc.reset(path=p)
        except Exception:
            pass


_set_show_hidden(show_hidden_cb.value)
show_hidden_cb.observe(lambda ch: _set_show_hidden(bool(ch["new"])), names="value")
jump_btn.on_click(lambda _: _jump_all(jump_path_tb.value))
jump_row = row(jump_path_tb, jump_btn)

# Generate widgets
kind_dd = widgets.Dropdown(
    options=[("ECDSA P-256", "ecdsa"), ("RSA-2048", "rsa")],
    value=LB_CERT_KIND,
    description="Kind:",
)
ecdsa_curve_dd = widgets.Dropdown(
    options=[("secp256r1 (prime256v1)", "secp256r1")],
    value=LB_ECDSA_CURVE,
    description="Curve:",
)
rsa_bits_dd = widgets.Dropdown(
    options=[("2048", 2048), ("3072", 3072)], value=LB_RSA_BITS, description="RSA bits:"
)
cn_in = widgets.Text(value=LB_CERT_CN, description="CN (SAN):")
days_in = widgets.BoundedIntText(
    value=int(LB_CERT_DAYS), min=1, max=36500, step=1, description="Days valid:"
)
outdir_in = widgets.Text(
    value=LB_PEM_OUTPUT_DIR,
    description="Output dir:",
    layout=widgets.Layout(width="100%"),
)
pem_basename_in = widgets.Text(
    value=os.environ.get("LB_PEM_BASENAME", "lb_current"),
    description="PEM basename:",
    layout=widgets.Layout(width="100%"),
)
overwrite_cb = widgets.Checkbox(
    value=False, description="Overwrite existing PEMs"
)  # NEW

gen_note = widgets.HTML(
    "<small>Generate writes PEMs locally. With determinstic basename it overwrites the same files; uncheck Overwrite to reuse existing without re-gen.</small>"
)

# Locust basics and endpoints
health_ep_in = widgets.Text(
    value=HEALTH_ENDPOINT_PATH,
    description="Health Path:",
    layout=widgets.Layout(width="100%"),
)
throughput_ep_in = widgets.Text(
    value=THROUGHPUT_ENDPOINT_PATH,
    description="Throughput Base Path (payload):",
    layout=widgets.Layout(width="100%"),
)
test_mode_dd = widgets.Dropdown(
    options=[
        ("CPS (connections/sec)", "cps"),
        ("Throughput (Gbps via payload)", "throughput"),
    ],
    value=TEST_MODE,
    description="Test Mode:",
)
wait_time_in = widgets.FloatText(
    value=LOCUST_WAIT_TIME_SEC, description="Wait(s)/user:", step=0.1
)
conn_timeout_in = widgets.BoundedIntText(
    value=LOCUST_CONNECT_TIMEOUT,
    min=1000,
    max=60000,
    step=500,
    description="Connect ms:",
)
read_timeout_in = widgets.BoundedIntText(
    value=LOCUST_READ_TIMEOUT, min=1000, max=120000, step=500, description="Read ms:"
)
verify_tls_in = widgets.Checkbox(
    value=LOCUST_VERIFY_TLS, description="Verify TLS (False recommended with VIP IP)"
)

# Single‑LB controls (caps only)
lb_min_in = widgets.BoundedIntText(
    value=int(LB_MIN_MBPS), min=10, max=32000, step=10, description="LB min (Mbps):"
)
lb_max_in = widgets.BoundedIntText(
    value=int(LB_MAX_MBPS), min=10, max=32000, step=10, description="LB max (Mbps):"
)
lb_count_label = widgets.HTML("<i>LB count is fixed to 1 for this Single‑LB test.</i>")

# UI params
ui_enable_in = widgets.Checkbox(value=UI_ENABLE, description="Enable Locust UI")
ui_mode_in = widgets.Dropdown(
    options=["tunnel", "nsg"], value=UI_EXPOSE_MODE, description="UI expose mode:"
)
ui_port_in = widgets.BoundedIntText(
    value=UI_WEB_PORT, min=1024, max=65535, step=1, description="UI port:"
)
ui_cidr_in = widgets.Text(value=UI_ALLOWED_CIDR, description="UI allowed CIDR:")

# CPU policy
workers_per_host_mode_in = widgets.Dropdown(
    options=[("Auto (use OCPUs)", "auto"), ("Fixed count", "fixed")],
    value="auto",
    description="Workers/host mode:",
)
fixed_workers_in = widgets.BoundedIntText(
    value=16, min=1, max=256, step=1, description="Fixed workers/host:"
)
cpu_reserve_in = widgets.BoundedIntText(
    value=int(CPU_RESERVE), min=0, max=8, step=1, description="CPU reserve:"
)
min_workers_in = widgets.BoundedIntText(
    value=int(MIN_WORKERS_PER_HOST),
    min=1,
    max=256,
    step=1,
    description="Min workers/host:",
)
max_workers_in = widgets.BoundedIntText(
    value=int(MAX_WORKERS_PER_HOST),
    min=1,
    max=512,
    step=1,
    description="Max workers/host:",
)

# Counts and shapes
backend_count_in = widgets.BoundedIntText(
    value=int(BACKEND_COUNT), min=1, max=64, step=1, description="Backends:"
)
generator_count_in = widgets.BoundedIntText(
    value=int(GENERATOR_COUNT), min=1, max=128, step=1, description="Generators:"
)


def _int_box(v, desc):
    return widgets.BoundedIntText(
        value=int(v), min=1, max=36000, step=1, description=desc
    )


# CPS tiers/durations
warm_10k_in, hold_10k_in = _int_box(CPS_WARMUPS[10000], "10k warmup(s):"), _int_box(
    CPS_HOLDS[10000], "10k hold(s):"
)
warm_25k_in, hold_25k_in = _int_box(CPS_WARMUPS[25000], "25k warmup(s):"), _int_box(
    CPS_HOLDS[25000], "25k hold(s):"
)
warm_35k_in, hold_35k_in = _int_box(CPS_WARMUPS[35000], "35k warmup(s):"), _int_box(
    CPS_HOLDS[35000], "35k hold(s):"
)
warm_50k_in, hold_50k_in = _int_box(CPS_WARMUPS[50000], "50k warmup(s):"), _int_box(
    CPS_HOLDS[50000], "50k hold(s):"
)
warm_100k_in, hold_100k_in = _int_box(CPS_WARMUPS[100000], "100k warmup(s):"), _int_box(
    CPS_HOLDS[100000], "100k hold(s):"
)

tput_targets_in = widgets.Text(
    value=TPUT_TARGETS_GBPS_TEXT,
    description="TPUT targets (Gbps):",
    layout=widgets.Layout(width="100%"),
)
tput_warm_in = widgets.BoundedIntText(
    value=int(TPUT_WARMUP_SEC), min=1, max=36000, step=1, description="TPUT warmup(s):"
)
tput_hold_in = widgets.BoundedIntText(
    value=int(TPUT_HOLD_SEC), min=1, max=36000, step=1, description="TPUT hold(s):"
)
tput_payload_size_in = widgets.Text(
    value=TPUT_PAYLOAD_SIZE_TEXT,
    description="TPUT payload for run (e.g., 4k, 10k, 100k, 1m):",
)
tput_payload_bake_in = widgets.Text(
    value=TPUT_PAYLOAD_SIZES_TEXT,
    description="Payload sizes to bake on backends (comma-separated):",
    layout=widgets.Layout(width="100%"),
)
payload_help = widgets.HTML(
    "<i>Suffixes: k ≈1000 bytes, m ≈1,000,000 bytes. Examples: 4k, 5k, 10k, 100k, 1m, 5m.</i>"
)

apply_btn = widgets.Button(
    description="Apply Selections",
    button_style="primary",
    icon="check",
    layout=widgets.Layout(width="240px", height="36px", align_self="center"),
)

_state = {
    "comp_dd": None,
    "ad_a_dd": None,
    "shape_filter": None,
    "backend_shape_dd": None,
    "generator_shape_dd": None,
    "image_dd": None,
    "backend_ocpus_in": None,
    "backend_mem_in": None,
    "generator_ocpus_in": None,
    "generator_mem_in": None,
}


def identity_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    signer = build_signer(
        TENANCY_OCID,
        USER_OCID,
        FINGERPRINT,
        OCI_PRIVATE_KEY_PATH,
        PRIVATE_KEY_PASSPHRASE,
        cfg_r,
    )
    return oci.identity.IdentityClient(config=cfg_r, signer=signer)


def compute_client_for_current():
    cfg_r = cfg_for_region(region_dd.value)
    signer = build_signer(
        TENANCY_OCID,
        USER_OCID,
        FINGERPRINT,
        OCI_PRIVATE_KEY_PATH,
        PRIVATE_KEY_PASSPHRASE,
        cfg_r,
    )
    return oci.core.ComputeClient(config=cfg_r, signer=signer)


def list_compartments(idc):
    comps = oci.pagination.list_call_get_all_results(
        idc.list_compartments,
        TENANCY_OCID,
        compartment_id_in_subtree=True,
        access_level="ACCESSIBLE",
    ).data
    comps = [c for c in comps if c.lifecycle_state == "ACTIVE"]
    tenancy = idc.get_tenancy(TENANCY_OCID).data
    tenancy_name = getattr(tenancy, "name", "root-tenancy")
    return [(f"{tenancy_name} (root)", TENANCY_OCID)] + sorted(
        [(c.name + f" ({c.description or 'no-desc'})", c.id) for c in comps],
        key=lambda t: t[0].lower(),
    )


def list_ads(idc):
    ads = oci.pagination.list_call_get_all_results(
        idc.list_availability_domains, TENANCY_OCID
    ).data
    return sorted([ad.name for ad in ads]) or ["AD-1"]


def list_shapes(cc):
    shapes = oci.pagination.list_call_get_all_results(cc.list_shapes, TENANCY_OCID).data
    return sorted({s.shape for s in shapes})


def list_images(cc, comp_id: str, b_shape: str, g_shape: str):
    def imgs_for(shape):
        return oci.pagination.list_call_get_all_results(
            cc.list_images,
            comp_id,
            operating_system="Oracle Linux",
            sort_by="TIMECREATED",
            sort_order="DESC",
            shape=shape,
        ).data

    if b_shape and g_shape:
        b, g = imgs_for(b_shape), imgs_for(g_shape)
        g_ids = {im.id for im in g}
        return [im for im in b if im.id in g_ids]
    shape = b_shape or g_shape
    return (
        imgs_for(shape)
        if shape
        else oci.pagination.list_call_get_all_results(
            cc.list_images,
            comp_id,
            operating_system="Oracle Linux",
            sort_by="TIMECREATED",
            sort_order="DESC",
        ).data
    )


def show_flex_inputs(back_shape: str, gen_shape: str):
    if _state["backend_ocpus_in"] is None:
        _state["backend_ocpus_in"] = widgets.BoundedIntText(
            value=int(BACKEND_OCPUS),
            min=1,
            max=128,
            step=1,
            description="Backend OCPUs:",
        )
        _state["backend_mem_in"] = widgets.BoundedIntText(
            value=int(BACKEND_MEMORY_GB),
            min=1,
            max=2048,
            step=1,
            description="Backend Memory(GB):",
        )
        _state["generator_ocpus_in"] = widgets.BoundedIntText(
            value=int(GENERATOR_OCPUS),
            min=1,
            max=256,
            step=1,
            description="Generator OCPUs:",
        )
        _state["generator_mem_in"] = widgets.BoundedIntText(
            value=int(GENERATOR_MEMORY_GB),
            min=1,
            max=4096,
            step=1,
            description="Generator Memory(GB):",
        )
    _state["backend_ocpus_in"].layout.display = (
        "block" if (back_shape or "").endswith(".Flex") else "none"
    )
    _state["backend_mem_in"].layout.display = (
        "block" if (back_shape or "").endswith(".Flex") else "none"
    )
    _state["generator_ocpus_in"].layout.display = (
        "block" if (gen_shape or "").endswith(".Flex") else "none"
    )
    _state["generator_mem_in"].layout.display = (
        "block" if (gen_shape or "").endswith(".Flex") else "none"
    )


def rebuild_dynamic_area(_=None):
    err_out.clear_output()
    with err_out:
        print(f"Refreshing for region: {region_dd.value} ...")
    try:
        idc = identity_client_for_current()
        cc = compute_client_for_current()

        comp_opts = list_compartments(idc)
        comp_dd = widgets.Dropdown(
            options=comp_opts,
            value=comp_opts[0][1],
            description="Compartment:",
            layout=widgets.Layout(width="100%"),
        )
        comp_filter = widgets.Text(
            value="",
            description="Comp filter:",
            placeholder="substring (optional)",
            layout=widgets.Layout(width="100%"),
        )

        def on_comp_filter_change(_ch):
            text = comp_filter.value.strip().lower()
            filtered = (
                comp_opts
                if not text
                else [o for o in comp_opts if text in o[0].lower()]
            )
            comp_dd.options = filtered or comp_opts
            comp_dd.value = (filtered or comp_opts)[0][1]
            refresh_images()

        comp_filter.observe(on_comp_filter_change, names="value")

        ad_names = list_ads(idc)
        ad_a_dd = widgets.Dropdown(
            options=[(n, n) for n in ad_names],
            value=ad_names[0],
            description="LB AD:",
            layout=widgets.Layout(width="100%"),
        )
        _state["comp_dd"], _state["ad_a_dd"] = comp_dd, ad_a_dd

        all_shapes = list_shapes(cc)
        shape_filter = widgets.Text(
            value="",
            description="Shape filter:",
            placeholder="e.g. E5.Flex",
            layout=widgets.Layout(width="100%"),
        )

        def filtered_shapes():
            if not shape_filter.value.strip():
                return all_shapes
            s = shape_filter.value.strip().lower()
            return [n for n in all_shapes if s in n.lower()]

        backend_shape_dd = widgets.Dropdown(
            options=[(n, n) for n in filtered_shapes()] or [("No shapes", "")],
            value=(filtered_shapes()[0] if filtered_shapes() else ""),
            description="Backend Shape:",
            layout=widgets.Layout(width="100%"),
        )
        generator_shape_dd = widgets.Dropdown(
            options=[(n, n) for n in filtered_shapes()] or [("No shapes", "")],
            value=(filtered_shapes()[0] if filtered_shapes() else ""),
            description="Generator Shape:",
            layout=widgets.Layout(width="100%"),
        )
        _state.update(
            {
                "shape_filter": shape_filter,
                "backend_shape_dd": backend_shape_dd,
                "generator_shape_dd": generator_shape_dd,
            }
        )

        def on_shape_filter(_ch):
            opts = filtered_shapes()
            backend_shape_dd.options = [(n, n) for n in opts] or [("No shapes", "")]
            backend_shape_dd.value = opts[0] if opts else ""
            generator_shape_dd.options = [(n, n) for n in opts] or [("No shapes", "")]
            generator_shape_dd.value = opts[0] if opts else ""
            show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value)
            refresh_images()

        shape_filter.observe(on_shape_filter, names="value")
        backend_shape_dd.observe(
            lambda _ch: (
                show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value),
                refresh_images(),
            ),
            names="value",
        )
        generator_shape_dd.observe(
            lambda _ch: (
                show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value),
                refresh_images(),
            ),
            names="value",
        )
        show_flex_inputs(backend_shape_dd.value, generator_shape_dd.value)

        image_dd = widgets.Dropdown(
            options=[("Select compartment first", "")],
            value="",
            description="Image:",
            layout=widgets.Layout(width="100%"),
        )
        _state["image_dd"] = image_dd

        def refresh_images():
            try:
                comp_id = comp_dd.value
                b, g = backend_shape_dd.value or "", generator_shape_dd.value or ""
                imgs = list_images(cc, comp_id, b, g)
                items = []
                for im in imgs[:100]:
                    try:
                        created = im.time_created.strftime("%Y-%m-%d")
                    except:
                        created = ""
                    items.append(
                        (
                            f"{im.display_name} — {im.operating_system} {im.operating_system_version} — {created}",
                            im.id,
                        )
                    )
                image_dd.options = items or [
                    ("No compatible Oracle Linux images for current shape(s))", "")
                ]
                image_dd.value = image_dd.options[0][1] if image_dd.options else ""
            except Exception:
                image_dd.options = [("Discovery error; try Reload/Reset", "")]
                image_dd.value = ""

        refresh_images()

        # Compose UI sections
        location_box = section(
            "Location",
            [row(comp_filter), vspace(), row(comp_dd), vspace(), row(ad_a_dd)],
        )
        shapes_box = section(
            "Shapes and Flex configuration",
            [
                row(shape_filter),
                vspace(),
                row(backend_shape_dd),
                vspace(),
                row(
                    backend_count_in,
                    _state["backend_ocpus_in"] or widgets.Label(""),
                    _state["backend_mem_in"] or widgets.Label(""),
                ),
                vspace(),
                row(generator_shape_dd),
                vspace(),
                row(
                    generator_count_in,
                    _state["generator_ocpus_in"] or widgets.Label(""),
                    _state["generator_mem_in"] or widgets.Label(""),
                ),
                vspace(),
                row(image_dd),
            ],
        )

        # Cert Mode: browse or generate
        browse_box = section(
            "LB TLS — Browse PEMs",
            [
                widgets.HTML(
                    "<i>Provide existing certificate and private key (optional CA chain).</i>"
                ),
                vspace("6px"),
                show_hidden_cb,
                row(jump_path_tb, jump_btn),
                vspace("6px"),
                fc_cert,
                vspace("6px"),
                fc_key,
                vspace("6px"),
                fc_ca,
            ],
        )
        generate_box = section(
            "LB TLS — Generate self‑signed PEMs",
            [
                row(kind_dd, ecdsa_curve_dd, rsa_bits_dd),
                vspace(),
                row(cn_in, days_in),
                vspace(),
                row(outdir_in),
                vspace(),
                row(pem_basename_in),
                row(overwrite_cb),  # NEW
                gen_note,
            ],
        )

        def _toggle_cert_ui(mode: str):
            m = (mode or "browse").lower()
            browse_box.layout.display = "block" if m == "browse" else "none"
            generate_box.layout.display = "block" if m == "generate" else "none"
            ecdsa_curve_dd.layout.display = (
                "block" if kind_dd.value == "ecdsa" and m == "generate" else "none"
            )
            rsa_bits_dd.layout.display = (
                "block" if kind_dd.value == "rsa" and m == "generate" else "none"
            )

        cert_mode_dd.observe(lambda ch: _toggle_cert_ui(ch["new"]), names="value")
        kind_dd.observe(lambda ch: _toggle_cert_ui(cert_mode_dd.value), names="value")
        _toggle_cert_ui(cert_mode_dd.value)

        lb_settings_box = section(
            "Load Balancer Settings (Single‑LB)",
            [
                widgets.HTML(
                    "<i>TLS is offloaded at the LB. PPv2 to backends on HTTP:80. Exactly one LB (one VIP).</i>"
                ),
                vspace("6px"),
                row(lb_min_in, lb_max_in),
                vspace("6px"),
                lb_count_label,
            ],
        )
        cert_mode_box = section("Certificate Mode", [cert_mode_dd])

        cps_box = section(
            "CPS settings (used when Test Mode=CPS)",
            [
                row(warm_10k_in, hold_10k_in),
                vspace(),
                row(warm_25k_in, hold_25k_in),
                vspace(),
                row(warm_35k_in, hold_35k_in),
                vspace(),
                row(warm_50k_in, hold_50k_in),
                vspace(),
                row(warm_100k_in, hold_100k_in),
            ],
        )
        tput_box = section(
            "Throughput settings (used when Test Mode=Throughput)",
            [
                row(tput_targets_in),
                vspace(),
                row(tput_warm_in, tput_hold_in),
                vspace(),
                row(tput_payload_size_in),
                vspace(),
                row(tput_payload_bake_in),
                payload_help,
            ],
        )
        cpu_box = section(
            "Generator CPU policy (workers per host)",
            [
                row(workers_per_host_mode_in, fixed_workers_in),
                vspace(),
                row(cpu_reserve_in, min_workers_in, max_workers_in),
                widgets.HTML(
                    "<i>Auto mode: workers/host = clamp(nproc − CPU_RESERVE, MIN, MAX).</i>"
                ),
            ],
        )
        cps_locust_box = section(
            "Locust & UI settings",
            [
                row(test_mode_dd),
                vspace(),
                row(wait_time_in, conn_timeout_in, read_timeout_in, verify_tls_in),
                vspace(),
                row(health_ep_in),
                vspace(),
                row(throughput_ep_in),
                vspace(),
                row(ui_enable_in, ui_mode_in, ui_port_in, ui_cidr_in),
            ],
        )

        dynamic_box.children = [
            location_box,
            shapes_box,
            lb_settings_box,
            cert_mode_box,
            browse_box,
            generate_box,
            cps_box,
            tput_box,
            cpu_box,
            cps_locust_box,
        ]
        err_out.clear_output()
    except Exception as e:
        dynamic_box.children = []
        with err_out:
            print("[error] UI rebuild failed:", repr(e))
            print(
                "Use Reset, then try again. If it persists, restart the kernel and run Cells 1–3."
            )


def on_reload_clicked(_):
    rebuild_dynamic_area()


def on_reset_clicked(_):
    region_dd.value = default_region
    rebuild_dynamic_area()


def _parse_size_text_to_bytes(s: str) -> tuple[int, str]:
    s = (s or "").strip().lower()
    if not s:
        raise ValueError("Empty payload size")
    if s.endswith("mb"):
        s = s[:-2] + "m"
    if s.endswith("kb"):
        s = s[:-2] + "k"
    if s.endswith("m"):
        n = float(s[:-1])
        if n <= 0:
            raise ValueError("MB value must be > 0")
        return int(n * 1_000_000), f"{int(n) if n.is_integer() else n}m"
    if s.endswith("k"):
        n = float(s[:-1])
        if n <= 0:
            raise ValueError("KB value must be > 0")
        return int(n * 1_000), f"{int(n) if n.is_integer() else n}k"
    n = float(s)
    if n <= 0:
        raise ValueError("Byte value must be > 0")
    return int(n), str(int(n))


def _normalize_bake_list(txt: str) -> list[str]:
    out = []
    for tok in (txt or "").split(","):
        tok = tok.strip()
        if not tok:
            continue
        _, label = _parse_size_text_to_bytes(tok)
        out.append(label)
    seen, norm = set(), []
    for l in out:
        if l in seen:
            continue
        seen.add(l)
        norm.append(l)
    return norm


def on_apply_clicked(_):
    try:
        global SELECTED_REGION, REGION, COMPARTMENT_ID, AD_A, AD_B, IMAGE_ID
        global SSH_PUBLIC_KEY_PATH, SSH_PRIVATE_KEY_PATH, SSH_PUBLIC_KEY_CONTENT
        global LB_CERT_PEM_PATH, LB_KEY_PEM_PATH, LB_CA_PEM_PATH, LB_CERT_MODE
        global BACKEND_COUNT, BACKEND_SHAPE, BACKEND_OCPUS, BACKEND_MEMORY_GB
        global GENERATOR_COUNT, GENERATOR_SHAPE, GENERATOR_OCPUS, GENERATOR_MEMORY_GB
        global LB_MIN_MBPS, LB_MAX_MBPS
        global HEALTH_ENDPOINT_PATH, THROUGHPUT_ENDPOINT_PATH
        global LOCUST_WAIT_TIME_SEC, LOCUST_CONNECT_TIMEOUT, LOCUST_READ_TIMEOUT, LOCUST_VERIFY_TLS
        global UI_ENABLE, UI_EXPOSE_MODE, UI_WEB_PORT, UI_ALLOWED_CIDR
        global TEST_MODE, CPS_WARMUPS, CPS_HOLDS
        global TPUT_TARGETS_GBPS_TEXT, TPUT_WARMUP_SEC, TPUT_HOLD_SEC
        global TPUT_PAYLOAD_SIZE_TEXT, TPUT_PAYLOAD_SIZE_BYTES, TPUT_PAYLOAD_SIZE_LABEL
        global TPUT_PAYLOAD_SIZES_TEXT, TPUT_PAYLOAD_BYTES_PER_REQ
        global LB_CERT_KIND, LB_ECDSA_CURVE, LB_RSA_BITS, LB_CERT_CN, LB_CERT_DAYS, LB_PEM_OUTPUT_DIR
        global LB_PEM_BASENAME

        SELECTED_REGION = region_dd.value
        REGION = SELECTED_REGION

        comp_dd = _state["comp_dd"]
        ad_a_dd = _state["ad_a_dd"]
        backend_shape_dd = _state["backend_shape_dd"]
        generator_shape_dd = _state["generator_shape_dd"]
        image_dd = _state["image_dd"]

        COMPARTMENT_ID = comp_dd.value
        AD_A = ad_a_dd.value
        AD_B = ad_a_dd.value  # Single subnet model

        BACKEND_COUNT = int(backend_count_in.value)
        GENERATOR_COUNT = int(generator_count_in.value)

        BACKEND_SHAPE = backend_shape_dd.value or ""
        GENERATOR_SHAPE = generator_shape_dd.value or ""
        if BACKEND_SHAPE.endswith(".Flex"):
            BACKEND_OCPUS = float(_state["backend_ocpus_in"].value)
            BACKEND_MEMORY_GB = float(_state["backend_mem_in"].value)
        if GENERATOR_SHAPE.endswith(".Flex"):
            GENERATOR_OCPUS = float(_state["generator_ocpus_in"].value)
            GENERATOR_MEMORY_GB = float(_state["generator_mem_in"].value)

        IMAGE_ID = image_dd.value or ""

        # SSH keys
        SSH_PUBLIC_KEY_PATH = os.path.expanduser(
            ssh_pub_dd.value or SSH_PUBLIC_KEY_PATH
        )
        SSH_PRIVATE_KEY_PATH = os.path.expanduser(
            ssh_priv_dd.value or SSH_PRIVATE_KEY_PATH
        )
        with open(SSH_PUBLIC_KEY_PATH, "r") as f:
            SSH_PUBLIC_KEY_CONTENT = f.read().strip()

        # LB Settings
        LB_MIN_MBPS = int(lb_min_in.value)
        LB_MAX_MBPS = int(lb_max_in.value)

        # Cert Mode handling
        LB_CERT_MODE = cert_mode_dd.value
        if LB_CERT_MODE == "browse":
            LB_CERT_PEM_PATH = os.path.expanduser(fc_cert.selected or "")
            LB_KEY_PEM_PATH = os.path.expanduser(fc_key.selected or "")
            LB_CA_PEM_PATH = os.path.expanduser(fc_ca.selected or "")
            if not LB_CERT_PEM_PATH or not os.path.exists(LB_CERT_PEM_PATH):
                raise ValueError(
                    "LB cert PEM missing. Use the 'Select LB certificate' picker."
                )
            if not LB_KEY_PEM_PATH or not os.path.exists(LB_KEY_PEM_PATH):
                raise ValueError(
                    "LB key PEM missing. Use the 'Select LB private key' picker."
                )
        else:
            LB_CERT_KIND = kind_dd.value
            LB_ECDSA_CURVE = ecdsa_curve_dd.value
            LB_RSA_BITS = int(rsa_bits_dd.value)
            LB_CERT_CN = (cn_in.value or "lb.local").strip()
            LB_CERT_DAYS = int(days_in.value)
            LB_PEM_OUTPUT_DIR = os.path.expanduser(outdir_in.value or "./local-lb-pems")
            LB_PEM_BASENAME = (pem_basename_in.value or "lb_current").strip()
            os.environ["LB_PEM_BASENAME"] = LB_PEM_BASENAME

            # Compute stable paths and decide to generate or reuse
            base_dir = LB_PEM_OUTPUT_DIR
            cert_path = os.path.join(base_dir, f"{LB_PEM_BASENAME}.cert.pem")
            key_path = os.path.join(base_dir, f"{LB_PEM_BASENAME}.key.pem")
            ca_path = os.path.join(base_dir, f"{LB_PEM_BASENAME}.ca.pem")

            exists_all = all(os.path.exists(p) for p in (cert_path, key_path, ca_path))
            if exists_all and not overwrite_cb.value:
                # Reuse existing stable files (no regeneration)
                LB_CERT_PEM_PATH, LB_KEY_PEM_PATH, LB_CA_PEM_PATH = (
                    cert_path,
                    key_path,
                    ca_path,
                )
            else:
                # Generate (overwrites stable files)
                cert_path, key_path, ca_path = generate_self_signed_lb_pems(
                    LB_CERT_KIND,
                    LB_ECDSA_CURVE,
                    LB_RSA_BITS,
                    LB_CERT_CN,
                    LB_CERT_DAYS,
                    LB_PEM_OUTPUT_DIR,
                )
                LB_CERT_PEM_PATH, LB_KEY_PEM_PATH, LB_CA_PEM_PATH = (
                    cert_path,
                    key_path,
                    ca_path,
                )

        # Persist PEM paths for Cell 6
        os.environ["LB_CERT_PEM_PATH"] = LB_CERT_PEM_PATH
        os.environ["LB_KEY_PEM_PATH"] = LB_KEY_PEM_PATH
        os.environ["LB_CA_PEM_PATH"] = LB_CA_PEM_PATH

        # Endpoints and timeouts
        HEALTH_ENDPOINT_PATH = (health_ep_in.value or "/healthz").strip()
        THROUGHPUT_ENDPOINT_PATH = (throughput_ep_in.value or "/payload_100k").strip()
        LOCUST_WAIT_TIME_SEC = float(wait_time_in.value)
        LOCUST_CONNECT_TIMEOUT = int(conn_timeout_in.value)
        LOCUST_READ_TIMEOUT = int(read_timeout_in.value)
        LOCUST_VERIFY_TLS = bool(verify_tls_in.value)

        # Payloads
        TPUT_TARGETS_GBPS_TEXT = (tput_targets_in.value or "1,5,10").strip()
        TPUT_WARMUP_SEC = int(tput_warm_in.value)
        TPUT_HOLD_SEC = int(tput_hold_in.value)
        TPUT_PAYLOAD_SIZE_TEXT = (tput_payload_size_in.value or "100k").strip()
        TPUT_PAYLOAD_SIZES_TEXT = (
            tput_payload_bake_in.value or "4k,5k,10k,50k,100k,256k,1m,5m"
        ).strip()

        def _parse_size_text_to_bytes_local(s: str):
            return _parse_size_text_to_bytes(s)

        TPUT_PAYLOAD_SIZE_BYTES, TPUT_PAYLOAD_SIZE_LABEL = (
            _parse_size_text_to_bytes_local(TPUT_PAYLOAD_SIZE_TEXT)
        )
        TPUT_PAYLOAD_BYTES_PER_REQ = int(TPUT_PAYLOAD_SIZE_BYTES)

        # UI toggles
        UI_ENABLE = bool(ui_enable_in.value)
        UI_EXPOSE_MODE = ui_mode_in.value or "tunnel"
        UI_WEB_PORT = int(ui_port_in.value)
        UI_ALLOWED_CIDR = (ui_cidr_in.value or "0.0.0.0/0").strip()

        # CPS tiers
        CPS_WARMUPS = {
            10000: int(warm_10k_in.value),
            25000: int(warm_25k_in.value),
            35000: int(warm_35k_in.value),
            50000: int(warm_50k_in.value),
            100000: int(warm_100k_in.value),
        }
        CPS_HOLDS = {
            10000: int(hold_10k_in.value),
            25000: int(hold_25k_in.value),
            35000: int(hold_35k_in.value),
            50000: int(hold_50k_in.value),
            100000: int(hold_100k_in.value),
        }

        # Persist env for downstream cells
        os.environ["OCI_CONFIG_FILE"] = OCI_CONFIG_FILE
        os.environ["OCI_PROFILE"] = OCI_PROFILE
        os.environ["REGION"] = REGION
        os.environ["COMPARTMENT_ID"] = COMPARTMENT_ID
        os.environ["LB_CERT_MODE"] = LB_CERT_MODE

        def _bn(p):
            try:
                return os.path.basename(p) if p else "(unset)"
            except:
                return "(unset)"

        summary_out.clear_output()
        with summary_out:
            display(
                HTML(
                    f"""
<div style="border:{BORDER};background:{SECTION_BG};padding:{PAD};">
  <b>Selections applied</b>
  <div style="font-family:ui-monospace; white-space:pre-wrap;">
Region: {REGION}
Compartment: {COMPARTMENT_ID}
AD: {AD_A}

LB (Single): min/max (Mbps): {LB_MIN_MBPS} / {LB_MAX_MBPS}
LB offload TLS: TCP listener with SSL; PPv2 to backends (HTTP:80)
Cert Mode: {LB_CERT_MODE}
PEM basename: {LB_PEM_BASENAME if LB_CERT_MODE=='generate' else '(n/a)'}
LB cert file: {_bn(LB_CERT_PEM_PATH)}
LB key file:  {_bn(LB_KEY_PEM_PATH)}
LB CA file:   {_bn(LB_CA_PEM_PATH)}

Backend Shape: {BACKEND_SHAPE} | OCPUs={BACKEND_OCPUS if BACKEND_SHAPE.endswith('.Flex') else '-'} | MemGB={BACKEND_MEMORY_GB if BACKEND_SHAPE.endswith('.Flex') else '-'} | Count={BACKEND_COUNT}
Generator Shape: {GENERATOR_SHAPE} | OCPUs={GENERATOR_OCPUS if GENERATOR_SHAPE.endswith('.Flex') else '-'} | MemGB={GENERATOR_MEMORY_GB if GENERATOR_SHAPE.endswith('.Flex') else '-'} | Count={GENERATOR_COUNT}
Image: {IMAGE_ID or '(unset)'}

Health path: {HEALTH_ENDPOINT_PATH}
Throughput path (selected payload): {THROUGHPUT_ENDPOINT_PATH}
Throughput payload (run): {TPUT_PAYLOAD_SIZE_LABEL} (~{TPUT_PAYLOAD_SIZE_BYTES} bytes)
Payloads to bake: {_normalize_bake_list(TPUT_PAYLOAD_SIZES_TEXT)}

Locust: wait(s)={LOCUST_WAIT_TIME_SEC} | connect(ms)={LOCUST_CONNECT_TIMEOUT} | read(ms)={LOCUST_READ_TIMEOUT} | verify_tls={LOCUST_VERIFY_TLS}
UI: enable={UI_ENABLE} | mode={UI_EXPOSE_MODE} | port={UI_WEB_PORT} | allowed_cidr={UI_ALLOWED_CIDR}

CPS Warmups: {CPS_WARMUPS}
CPS Holds:   {CPS_HOLDS}
  </div>
</div>
"""
                )
            )
        print("Cell 3 complete: Selections applied; PEMs ready for Cell 6.")
        print(
            "- PROVISION infra: run Cells 4 → 7. Then Cells 8 → 10. Orchestration in Cells 11–12."
        )
    except Exception as e:
        err_out.clear_output()
        with err_out:
            print("[error] Apply failed:", repr(e))


reload_btn.on_click(on_reload_clicked)
reset_btn.on_click(on_reset_clicked)
apply_btn.on_click(on_apply_clicked)
region_dd.observe(rebuild_dynamic_area, names="value")

top_region = section("Region", [row(region_dd, reload_btn, reset_btn)])
rebuild_dynamic_area()
container = widgets.VBox(
    [
        top_region,
        dynamic_box,
        section("Keys (API + SSH)", [row(ssh_pub_dd), vspace(), row(ssh_priv_dd)]),
        section("Apply", [apply_btn]),
        err_out,
        summary_out,
    ],
    layout=widgets.Layout(width="100%", max_width=CONTAINER_MAX_W, margin="0 auto"),
)
display(container)

print("Cell 3 loaded: Choose Cert Mode (Browse/Generate), complete inputs, then Apply.")

In [ ]:
# Cell 4 — Objective: Write cloud-init scripts (Backends HTTP:80 proxy_protocol; Generators Locust; payloads baked)

import os

os.makedirs("cloud-init", exist_ok=True)


def _dd_lines_for_payloads(sizes_csv: str) -> str:
    lines = []

    def _one(tok: str):
        tok = tok.strip().lower()
        if not tok:
            return
        if tok.endswith("mb"):
            tok2 = tok[:-2] + "m"
        elif tok.endswith("kb"):
            tok2 = tok[:-2] + "k"
        else:
            tok2 = tok
        if tok2.endswith("m"):
            n = tok2[:-1]
            lines.append(
                f"dd if=/dev/zero of=/usr/share/nginx/html/payload_{n}m bs=1MB count={n} status=none || true"
            )
        elif tok2.endswith("k"):
            n = tok2[:-1]
            lines.append(
                f"dd if=/dev/zero of=/usr/share/nginx/html/payload_{n}k bs=1KB count={n} status=none || true"
            )
        else:
            lines.append(
                f"head -c {tok2} /dev/zero > /usr/share/nginx/html/payload_{tok2}b || true"
            )

    for t in (TPUT_PAYLOAD_SIZES_TEXT or "").split(","):
        _one(t)
    if not lines:
        lines.append(
            "dd if=/dev/zero of=/usr/share/nginx/html/payload_100k bs=1KB count=100 status=none || true"
        )
    return "\n".join(lines)


_dd_payload_block = _dd_lines_for_payloads(TPUT_PAYLOAD_SIZES_TEXT)


backend_cloud_init_template = """#!/bin/bash
if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi
if systemctl list-unit-files | grep -q oracle-cloud-agent.service; then
  systemctl enable --now oracle-cloud-agent || true
fi

# Kernel tuning for high CPS
cat <<EOF >/etc/sysctl.d/99-net-tuning.conf
net.core.somaxconn=262144
net.core.netdev_max_backlog=500000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
net.ipv4.tcp_tw_reuse=1
net.core.rmem_max=33554432
net.core.wmem_max=33554432
net.ipv4.tcp_syncookies=1
net.ipv4.tcp_fastopen=3
# Additional safe CPS-focused tweaks
net.ipv4.tcp_abort_on_overflow=0
net.ipv4.tcp_synack_retries=2
net.ipv4.tcp_syn_retries=3
fs.file-max=1000000
EOF
sysctl --system || true
echo "* - nofile 1048576" | tee -a /etc/security/limits.conf


dnf -y makecache || true
dnf -y install nginx || true

# Global NGINX tuning
cat >/etc/nginx/nginx.conf <<'NGINX'
user  nginx;
worker_processes auto;
worker_rlimit_nofile 1048576;

events { worker_connections 131072; multi_accept on; accept_mutex off; }

http {
    include /etc/nginx/mime.types;
    default_type application/octet-stream;

    # I/O and TCP toggles
    sendfile on; tcp_nopush on; tcp_nodelay on;

    # Short timeouts (CPS mode uses Connection: close)
    keepalive_timeout 15;

    # Tighten idle/slow clients; reset timed out connections early for CPS
    client_body_timeout 10; client_header_timeout 10; send_timeout 10;
    reset_timedout_connection on;

    # Large keepalive request cap (harmless for CPS; useful if switching to keepalive)
    keepalive_requests 100000;

    server_tokens off; access_log off;

    include /etc/nginx/conf.d/*.conf;
}
NGINX

# Pre-create payload files for throughput
__DD_PAYLOAD_BLOCK__

# NGINX server: HTTP:80 with PROXY protocol v2 expected from LB
cat >/etc/nginx/conf.d/freewheel.conf <<'EOF'
server {
    listen 80 proxy_protocol reuseport backlog=262144 fastopen=512;
    server_name _;
    access_log off;

    # Trust the LB subnet (adjust if your LB subnet differs)
    set_real_ip_from 10.0.1.0/24;
    real_ip_header proxy_protocol;
    real_ip_recursive on;

    location = /healthz { return 200 "ok\n"; }
    location /payload_ { root /usr/share/nginx/html; }
    location /          { return 200 "ok\n"; }
}
EOF

rm -f /etc/nginx/conf.d/default.conf
nginx -t && systemctl enable --now nginx
"""

backend_cloud_init = backend_cloud_init_template.replace(
    "__DD_PAYLOAD_BLOCK__", _dd_payload_block
)

generator_cloud_init = r"""#!/bin/bash
if command -v firewall-cmd >/dev/null 2>&1; then
  systemctl stop firewalld || true
  systemctl disable firewalld || true
fi
cat <<EOF >/etc/sysctl.d/99-freewheel.conf
net.core.somaxconn=65535
net.core.netdev_max_backlog=250000
net.ipv4.tcp_max_syn_backlog=262144
net.ipv4.ip_local_port_range=1024 65535
net.ipv4.tcp_fin_timeout=15
net.ipv4.tcp_tw_reuse=1
fs.file-max=1000000
EOF
sysctl --system || true
echo "* - nofile 1048576" >> /etc/security/limits.conf


dnf -y makecache || true
dnf -y install python3 python3-pip tmux curl || true
pip3 install --no-cache-dir --upgrade pip || true
pip3 install --no-cache-dir locust || true

mkdir -p /home/opc/locustwork/results
chown -R opc:opc /home/opc/locustwork

cat > /home/opc/locustwork/locustfile.py <<'PY'
import os
from locust import HttpUser, task, constant

VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()  # "cps" or "throughput"
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")

class CpsUser(HttpUser):
    wait_time = constant(WAIT_TIME_S)

    @task
    def do_request(self):
        path    = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        headers = {"Connection": "close"} if MODE == "cps" else {}  # keep-alive by default for throughput
        self.client.get(
            path,
            headers=headers,
            verify=VERIFY_TLS,
            timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
            name=("cps_req" if MODE == "cps" else "throughput_req"),
        )
PY

chown -R opc:opc /home/opc/locustwork
echo READY
"""

with open("cloud-init/backend.sh.tftpl", "w") as f:
    f.write(backend_cloud_init)
with open("cloud-init/generator.sh", "w") as f:
    f.write(generator_cloud_init)

print(
    "Cell 4 complete: Wrote backend.sh.tftpl (HTTP:80 + PPv2, tuned) and generator.sh (Locust; payloads baked)."
)
print(
    "NEXT: Run Cell 5 to generate Terraform (TLS offload + PPv2; TCP/80 health), then Cell 6 tfvars, then Cell 7 apply."
)

In [ ]:
# Cell 5 — Objective: Terraform using NSGs, VCN/subnets, Single private LB with TCP listener + TLS offload + PPv2 (backends on HTTP:80)
# Fully stateless posture while keeping the Default Security List:
# - Default Security List is managed and set to stateless allow-all (egress+ingress)
# - NSG rules are stateless with symmetric return-path
# - Subnets use the default SL (no security_list_ids override)

single_ad = (AD_A == AD_B) or (str(AD_B or "").strip() == "")

backend_shape_config_block = ""
if (BACKEND_SHAPE or "").endswith(".Flex"):
    backend_shape_config_block = """
  shape_config {
    ocpus         = %s
    memory_in_gbs = %s
  }""" % (
        BACKEND_OCPUS,
        BACKEND_MEMORY_GB,
    )

generator_shape_config_block = ""
if (GENERATOR_SHAPE or "").endswith(".Flex"):
    generator_shape_config_block = """
  shape_config {
    ocpus         = %s
    memory_in_gbs = %s
  }""" % (
        GENERATOR_OCPUS,
        GENERATOR_MEMORY_GB,
    )

terraform_config_template = """terraform {
  required_providers {
    oci = {
      source  = "oracle/oci"
      version = ">= 5.39.0"
    }
  }
}

provider "oci" {
  config_file_profile = var.oci_profile
  region              = var.region
}

# Variables
variable "oci_profile" {}
variable "region" {}
variable "compartment_id" {}
variable "ad_a" {}
variable "ad_b" {}
variable "image_id" {}
variable "ssh_public_key_content" {}
variable "ssh_private_key_path" {}

variable "backend_count" {}
variable "backend_shape" {}
variable "generator_count" {}
variable "generator_shape" {}

variable "ssh_allowed_cidr" {}

# TLS PEMs for LB offload (Cell 6 embeds contents)
variable "lb_cert_pem" { type = string }
variable "lb_key_pem"  { type = string }
variable "lb_ca_pem"   { type = string }

variable "lb_min_mbps" {}
variable "lb_max_mbps" {}

variable "bastion_plugin_name" {
  type    = string
  default = "Bastion"
}

# VCN
resource "oci_core_vcn" "vcn" {
  cidr_block     = "10.0.0.0/16"
  compartment_id = var.compartment_id
  display_name   = "cps-singlelb-vcn"
}

# Manage the Default Security List for the VCN and make it stateless allow-all
resource "oci_core_default_security_list" "default_sl" {
  manage_default_resource_id = oci_core_vcn.vcn.default_security_list_id

  egress_security_rules {
    protocol         = "all"
    destination      = "0.0.0.0/0"
    destination_type = "CIDR_BLOCK"
    stateless        = true
  }

  ingress_security_rules {
    protocol   = "all"
    source     = "0.0.0.0/0"
    source_type = "CIDR_BLOCK"
    stateless  = true
  }
}

# Gateways
resource "oci_core_internet_gateway" "igw" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "cps-singlelb-igw"
}
resource "oci_core_nat_gateway" "nat" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "cps-singlelb-nat"
}

# Route Tables
resource "oci_core_route_table" "rt_public" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "cps-singlelb-rt-public"
  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_internet_gateway.igw.id
  }
}
resource "oci_core_route_table" "rt_private" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "cps-singlelb-rt-private"
  route_rules {
    destination       = "0.0.0.0/0"
    destination_type  = "CIDR_BLOCK"
    network_entity_id = oci_core_nat_gateway.nat.id
  }
}

# NSGs
resource "oci_core_network_security_group" "nsg_lb" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-lb"
}
resource "oci_core_network_security_group" "nsg_backends" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-backends"
}
resource "oci_core_network_security_group" "nsg_generators" {
  compartment_id = var.compartment_id
  vcn_id         = oci_core_vcn.vcn.id
  display_name   = "nsg-generators"
}

# NSG Rules (ALL STATELESS with symmetric return-path rules)

# LB: allow HTTPS/TCP 443 from generators (ingress)
resource "oci_core_network_security_group_security_rule" "lb_ingress_443_from_generators" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 443
      max = 443
    }
  }
}

# LB: egress HTTP 80 to backends
resource "oci_core_network_security_group_security_rule" "lb_egress_80_to_backends" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "NETWORK_SECURITY_GROUP"
  destination               = oci_core_network_security_group.nsg_backends.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

# LB (stateless return): ingress from backends with source port 80 (backend -> LB response)
resource "oci_core_network_security_group_security_rule" "lb_ingress_from_backends_src80" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_backends.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 80
      max = 80
    }
  }
}

# LB (stateless return): egress to generators with source port 443 (LB -> client response)
resource "oci_core_network_security_group_security_rule" "lb_egress_to_generators_src443" {
  network_security_group_id = oci_core_network_security_group.nsg_lb.id
  direction                 = "EGRESS"
  protocol                  = "6"
  destination_type          = "NETWORK_SECURITY_GROUP"
  destination               = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 443
      max = 443
    }
  }
}

# Backends: allow HTTP 80 from LB (ingress)
resource "oci_core_network_security_group_security_rule" "backends_ingress_80_from_lb" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_lb.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 80
      max = 80
    }
  }
}

# Backends: egress all (package fetch, responses)
resource "oci_core_network_security_group_security_rule" "backends_egress_all" {
  network_security_group_id = oci_core_network_security_group.nsg_backends.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

# Generators: ingress SSH from CIDR
resource "oci_core_network_security_group_security_rule" "gens_ingress_ssh" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "CIDR_BLOCK"
  source                    = var.ssh_allowed_cidr
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 22
      max = 22
    }
  }
}

# Generators: intra-NSG for locust master ports 5557-5558 (ingress)
resource "oci_core_network_security_group_security_rule" "gens_ingress_locust_master" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    destination_port_range {
      min = 5557
      max = 5558
    }
  }
}

# Generators (stateless return): allow responses from Locust master ports 5557-5558 (source port match)
resource "oci_core_network_security_group_security_rule" "gens_ingress_from_generators_src5557_5558" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "INGRESS"
  protocol                  = "6"
  source_type               = "NETWORK_SECURITY_GROUP"
  source                    = oci_core_network_security_group.nsg_generators.id
  stateless                 = true
  tcp_options {
    source_port_range {
      min = 5557
      max = 5558
    }
  }
  description = "Stateless return path for Locust ports 5557-5558"
}

# Generators: egress all (to LB/Internet)
resource "oci_core_network_security_group_security_rule" "gens_egress_all" {
  network_security_group_id = oci_core_network_security_group.nsg_generators.id
  direction                 = "EGRESS"
  protocol                  = "all"
  destination_type          = "CIDR_BLOCK"
  destination               = "0.0.0.0/0"
  stateless                 = true
}

# Subnets use the DEFAULT (now stateless) Security List
resource "oci_core_subnet" "lb_priv" {
  cidr_block                 = "10.0.1.0/24"
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "cps-singlelb-lb-priv"
  route_table_id             = oci_core_route_table.rt_private.id
  prohibit_public_ip_on_vnic = true
}
resource "oci_core_subnet" "backends_priv" {
  cidr_block                 = "10.0.2.0/24"
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "cps-singlelb-backends-priv"
  route_table_id             = oci_core_route_table.rt_private.id
  prohibit_public_ip_on_vnic = true
}
resource "oci_core_subnet" "gens_pub" {
  cidr_block                 = "10.0.3.0/24"
  compartment_id             = var.compartment_id
  vcn_id                     = oci_core_vcn.vcn.id
  display_name               = "cps-singlelb-generators-pub"
  route_table_id             = oci_core_route_table.rt_public.id
  prohibit_public_ip_on_vnic = false
}

# Compute: backends (HTTP:80 + proxy_protocol expected)
resource "oci_core_instance" "backend" {
  count               = var.backend_count
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.backend_shape
__BACKEND_SHAPE_CONFIG__
  source_details {
    source_type = "image"
    source_id   = var.image_id
  }
  create_vnic_details {
    subnet_id        = oci_core_subnet.backends_priv.id
    assign_public_ip = false
    nsg_ids          = [oci_core_network_security_group.nsg_backends.id]
  }
  agent_config {
    are_all_plugins_disabled = false
    is_management_disabled   = false
    is_monitoring_disabled   = false
    plugins_config {
      name          = var.bastion_plugin_name
      desired_state = "ENABLED"
    }
  }
  display_name = "cps-backend-${count.index}"
  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content
    user_data           = base64encode(file("${path.module}/cloud-init/backend.sh.tftpl"))
  }
}

# Compute: generators
resource "oci_core_instance" "generator" {
  count               = var.generator_count
  availability_domain = var.ad_a
  compartment_id      = var.compartment_id
  shape               = var.generator_shape
__GENERATOR_SHAPE_CONFIG__
  source_details {
    source_type = "image"
    source_id   = var.image_id
  }
  create_vnic_details {
    subnet_id        = oci_core_subnet.gens_pub.id
    assign_public_ip = true
    nsg_ids          = [oci_core_network_security_group.nsg_generators.id]
  }
  display_name = "cps-gen-${count.index}"
  metadata = {
    ssh_authorized_keys = var.ssh_public_key_content
    user_data           = filebase64("${path.module}/cloud-init/generator.sh")
  }
}

# Flexible LB (private, TLS offload on listener)
resource "oci_load_balancer_load_balancer" "lb" {
  compartment_id = var.compartment_id
  display_name   = "cps-flex-lb"
  shape          = "flexible"
  is_private     = true
  subnet_ids     = [oci_core_subnet.lb_priv.id]
  network_security_group_ids = [oci_core_network_security_group.nsg_lb.id]
  shape_details {
    minimum_bandwidth_in_mbps = var.lb_min_mbps
    maximum_bandwidth_in_mbps = var.lb_max_mbps
  }
}

# LB certificate (Single-LB)
resource "oci_load_balancer_certificate" "lb_cert" {
  load_balancer_id   = oci_load_balancer_load_balancer.lb.id
  certificate_name   = "lb-cert"
  public_certificate = var.lb_cert_pem
  private_key        = var.lb_key_pem
  ca_certificate     = var.lb_ca_pem
}

# Backend set (TCP health check on 80)
resource "oci_load_balancer_backend_set" "bs" {
  load_balancer_id = oci_load_balancer_load_balancer.lb.id
  name             = "backendset1"
  policy           = "ROUND_ROBIN"
  health_checker {
    protocol          = "TCP"
    port              = 80
    retries           = 3
    interval_ms       = 10000
    timeout_in_millis = 5000
  }
}

# Attach all backends (port 80)
resource "oci_load_balancer_backend" "backends" {
  count            = var.backend_count
  load_balancer_id = oci_load_balancer_load_balancer.lb.id
  backendset_name  = oci_load_balancer_backend_set.bs.name
  ip_address       = oci_core_instance.backend[count.index].private_ip
  port             = 80
  weight           = 1
}

# TCP listener with SSL offload and Proxy Protocol v2 (TLS 1.3 + 1.2, server order, modern TLS 1.2 suite)
resource "oci_load_balancer_listener" "tcp443" {
  load_balancer_id         = oci_load_balancer_load_balancer.lb.id
  name                     = "tcp-443"
  port                     = 443
  protocol                 = "TCP"
  default_backend_set_name = oci_load_balancer_backend_set.bs.name

  ssl_configuration {
    certificate_name        = oci_load_balancer_certificate.lb_cert.certificate_name
    verify_peer_certificate = false
    protocols               = ["TLSv1.3", "TLSv1.2"]
    server_order_preference = "ENABLED"
    cipher_suite_name       = "oci-modern-ssl-cipher-suite-v1"
  }

  connection_configuration {
    idle_timeout_in_seconds            = 1200
    backend_tcp_proxy_protocol_version = 2
  }

  depends_on = [oci_load_balancer_certificate.lb_cert]
}

# Outputs
output "lb_id"               { value = oci_load_balancer_load_balancer.lb.id }
output "lb_ip_addresses"     { value = [for d in oci_load_balancer_load_balancer.lb.ip_address_details : d.ip_address] }
output "gen_public_ips"      { value = [for i in oci_core_instance.generator : i.public_ip] }
output "backend_private_ips" { value = [for i in oci_core_instance.backend   : i.private_ip] }
"""

terraform_config = terraform_config_template.replace(
    "__BACKEND_SHAPE_CONFIG__", backend_shape_config_block
).replace("__GENERATOR_SHAPE_CONFIG__", generator_shape_config_block)

with open("main.tf", "w") as f:
    f.write(terraform_config)

print(
    "Cell 5 updated: Default Security List kept and made stateless (allow-all); all NSG rules stateless with return-path."
)

In [ ]:
# Cell 6 — Objective: Write terraform.tfvars including LB PEMs (no secrets in code; read from env paths)
# Reads LB_CERT_PEM_PATH / LB_KEY_PEM_PATH / LB_CA_PEM_PATH from environment (exported by Cell 3).

import os


def _read_text_required(path: str, label: str) -> str:
    p = os.path.expanduser(path or "")
    if not p or not os.path.exists(p):
        raise ValueError(
            f"{label} missing. Set via Cell 3. Current: {path or '(unset)'}"
        )
    with open(p, "r") as f:
        return f.read().strip()


def _read_text_optional(path: str) -> str:
    p = os.path.expanduser(path or "")
    if not p or not os.path.exists(p):
        return ""
    with open(p, "r") as f:
        return f.read().strip()


# Validate required selections from Cell 3
if not IMAGE_ID:
    raise ValueError("IMAGE_ID is not set. Run Cell 3 and select an Image.")
if not BACKEND_SHAPE:
    raise ValueError("BACKEND_SHAPE is not set. Run Cell 3 and select a Backend Shape.")
if not GENERATOR_SHAPE:
    raise ValueError(
        "GENERATOR_SHAPE is not set. Run Cell 3 and select a Generator Shape."
    )

if not SSH_PUBLIC_KEY_PATH or not os.path.exists(SSH_PUBLIC_KEY_PATH):
    raise ValueError(
        f"SSH public key missing. Pick a valid key in Cell 3. Current: {SSH_PUBLIC_KEY_PATH}"
    )
if not SSH_PRIVATE_KEY_PATH or not os.path.exists(SSH_PRIVATE_KEY_PATH):
    raise ValueError(
        f"SSH private key missing. Pick a valid key in Cell 3. Current: {SSH_PRIVATE_KEY_PATH}"
    )

# Load SSH public key content
with open(os.path.expanduser(SSH_PUBLIC_KEY_PATH), "r") as f:
    ssh_public_key_content = f.read().strip()

# Read PEMs from paths persisted in Cell 3 (either browsed or generated)
LB_CERT_PEM_PATH = os.path.expanduser(os.environ.get("LB_CERT_PEM_PATH", ""))
LB_KEY_PEM_PATH = os.path.expanduser(os.environ.get("LB_KEY_PEM_PATH", ""))
LB_CA_PEM_PATH = os.path.expanduser(os.environ.get("LB_CA_PEM_PATH", ""))  # optional

lb_cert_pem = _read_text_required(LB_CERT_PEM_PATH, "LB cert PEM")
lb_key_pem = _read_text_required(LB_KEY_PEM_PATH, "LB key PEM")
lb_ca_pem = _read_text_optional(LB_CA_PEM_PATH)  # may be blank


tfvars = f"""
oci_profile            = "{OCI_PROFILE}"
region                 = "{REGION}"
compartment_id         = "{COMPARTMENT_ID}"
ad_a                   = "{AD_A}"
ad_b                   = "{AD_B}"
image_id               = "{IMAGE_ID}"
ssh_public_key_content = "{ssh_public_key_content}"
ssh_private_key_path   = "{os.path.expanduser(SSH_PRIVATE_KEY_PATH)}"

backend_count   = {int(BACKEND_COUNT)}
backend_shape   = "{BACKEND_SHAPE}"

generator_count = {int(GENERATOR_COUNT)}
generator_shape = "{GENERATOR_SHAPE}"

ssh_allowed_cidr = "{SSH_ALLOWED_CIDR}"
lb_min_mbps      = {int(LB_MIN_MBPS)}
lb_max_mbps      = {int(LB_MAX_MBPS)}

# LB TLS PEM material (strings)
lb_cert_pem = <<EOCERT
{lb_cert_pem}
EOCERT
lb_key_pem  = <<EOKEY
{lb_key_pem}
EOKEY
lb_ca_pem   = <<EOCA
{lb_ca_pem}
EOCA

bastion_plugin_name = "Bastion"
""".lstrip()

with open("terraform.tfvars", "w") as f:
    f.write(tfvars)


def _bn(p: str) -> str:
    try:
        return os.path.basename(p) if p else "(unset)"
    except:
        return "(unset)"


print("Cell 6 complete: terraform.tfvars written.")
print("Summary:")
print(f"  oci_profile={OCI_PROFILE} region={REGION}")
print(f"  ad_a={AD_A} ad_b={AD_B}")
print(
    f"  backend_shape={BACKEND_SHAPE} x{BACKEND_COUNT}  generator_shape={GENERATOR_SHAPE} x{GENERATOR_COUNT}"
)
print(f"  LB caps: min/max Mbps = {LB_MIN_MBPS}/{LB_MAX_MBPS}")
print(
    f"  LB cert={_bn(LB_CERT_PEM_PATH)}  key={_bn(LB_KEY_PEM_PATH)}  ca={_bn(LB_CA_PEM_PATH) or '(empty)'}"
)
print("NEXT: Run Cell 7 to terraform init/apply.")

In [ ]:
# Cell 7 — Objective: Initialize and apply Terraform (fresh build)

!terraform init
!terraform apply -auto-approve -var-file=terraform.tfvars

print("Cell 7 complete: Terraform apply finished.")
print("NEXT: Run Cell 8 to capture outputs (LB/Generators/Backends).")


In [ ]:
# Cell 8 — Objective: Extract LB/instance outputs and persist them for downstream cells

import os, json, subprocess


def _tf_output_json() -> dict:
    r = subprocess.run(["terraform", "output", "-json"], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"terraform output failed:\n{r.stderr}")
    try:
        return json.loads(r.stdout or "{}")
    except json.JSONDecodeError as e:
        raise RuntimeError(f"Could not parse terraform output JSON: {e}")


def _val(outputs: dict, key: str):
    obj = outputs.get(key, {})
    # Handle both { "value": ... } and raw values just in case
    return obj.get("value", obj if obj else None)


outputs = _tf_output_json()

LB_ID = _val(outputs, "lb_id")
LB_IPS = _val(outputs, "lb_ip_addresses") or []
GEN_IPS = _val(outputs, "gen_public_ips") or []
BACKEND_IPS = _val(outputs, "backend_private_ips") or []

if not LB_ID:
    raise RuntimeError("Missing LB_ID in Terraform outputs.")
if not isinstance(LB_IPS, list) or len(LB_IPS) < 1:
    raise RuntimeError(
        f"Missing VIP(s) in Terraform outputs (lb_ip_addresses): {LB_IPS}"
    )

# Single‑LB: select the single VIP
TARGET_HOST = LB_IPS[0]

# Persist for later cells and external tools (best‑effort)
os.environ["LB_ID"] = str(LB_ID)
os.environ["TARGET_HOST"] = str(TARGET_HOST)
os.environ["TARGET_URL"] = f"https://{TARGET_HOST}"

# Save a snapshot of outputs
snap = {
    "lb_id": LB_ID,
    "lb_ips": LB_IPS,
    "gen_public_ips": GEN_IPS,
    "backend_private_ips": BACKEND_IPS,
    "target_host": TARGET_HOST,
    "target_url": f"https://{TARGET_HOST}",
}
out_dir = OUTPUT_DIR if "OUTPUT_DIR" in globals() else "./results"
os.makedirs(out_dir, exist_ok=True)
snap_path = os.path.join(out_dir, f"terraform_outputs_{TS_UTC}.json")
with open(snap_path, "w") as f:
    json.dump(snap, f, indent=2)

print("LB_ID:", LB_ID)
print("LB_IPS:", LB_IPS)
print("Generators:", GEN_IPS)
print("Backends:", BACKEND_IPS)
print(f"TARGET_HOST (VIP): {TARGET_HOST}")
print(f"Saved outputs snapshot: {snap_path}")

print("Cell 8 complete: Outputs captured and persisted.")
print("NEXT: Run Cell 9 for SSH + Monitoring helpers.")

In [ ]:
# Cell 9 — Objective: SSH helpers, LB bandwidth update, Monitoring helpers

import os, time, json, re
import paramiko
import numpy as np
import pandas as pd
from datetime import datetime, timezone, timedelta
import oci

SSH_KEY_PASSPHRASE = os.environ.get("SSH_KEY_PASSPHRASE", None)
LB_METRICS_NAMESPACE = "oci_lbaas"


def _load_pkey(key_path: str, passphrase: str | None):
    kpath = os.path.expanduser(key_path)
    if not os.path.exists(kpath):
        raise FileNotFoundError(f"SSH private key not found: {kpath}")
    excs = []
    for KeyClass in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return KeyClass.from_private_key_file(kpath, password=passphrase)
        except Exception as e:
            excs.append(repr(e))
    raise paramiko.SSHException(
        "Could not load SSH private key. Path: %s\n - %s" % (kpath, "\n - ".join(excs))
    )


def ssh_exec(
    host, user="opc", key_path=SSH_PRIVATE_KEY_PATH, command="echo ok", timeout=600
):
    pkey = _load_pkey(key_path, SSH_KEY_PASSPHRASE)
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username=user, pkey=pkey, timeout=30)
    stdin, stdout, stderr = ssh.exec_command(command, timeout=timeout)
    try:
        out = stdout.read().decode("utf-8", errors="ignore")
    except Exception:
        out = ""
    try:
        err = stderr.read().decode("utf-8", errors="ignore")
    except Exception:
        err = ""
    try:
        rc = stdout.channel.recv_exit_status()
    except Exception:
        rc = None
    ssh.close()
    return out.strip(), err.strip(), rc


# Non-blocking launcher for nohup/tmux commands (prevents Paramiko read timeouts)
def ssh_exec_quick(host, command):
    pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, SSH_KEY_PASSPHRASE)
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    transport = ssh.get_transport()
    chan = transport.open_session()
    chan.exec_command(command)
    # do not read stdout/stderr; do not wait for exit
    time.sleep(0.5)
    try:
        chan.close()
    except Exception:
        pass
    ssh.close()


def ssh_exec_many(hosts, cmd, timeout=1200):
    return {h: ssh_exec(h, command=cmd, timeout=timeout) for h in (hosts or [])}


cfg = oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
cfg["region"] = REGION
lbc = oci.load_balancer.LoadBalancerClient(cfg)

_lb = lbc.get_load_balancer(LB_ID).data if LB_ID else None
COMPARTMENT_ID = (
    getattr(_lb, "compartment_id", os.environ.get("COMPARTMENT_ID", "")) or ""
)
mon = oci.monitoring.MonitoringClient(cfg)


def list_available_lb_metrics(lb_id: str):
    try:
        details = oci.monitoring.models.ListMetricsDetails(
            namespace=LB_METRICS_NAMESPACE,
            group_by=["name"],
            dimension_filters={"resourceId": lb_id},
        )
        resp = mon.list_metrics(
            compartment_id=COMPARTMENT_ID, list_metrics_details=details
        )
        names = sorted({m.name for m in resp.data if getattr(m, "name", None)})
        if names:
            return names
    except Exception as e:
        print("list_metrics failed; using fallback set. Reason:", repr(e))
    return sorted(
        list(
            {
                "NewConnections",
                "AcceptedConnections",
                "HandledConnections",
                "ActiveConnections",
                "ClosedConnections",
                "BytesIn",
                "BytesOut",
                "BytesReceived",
                "BytesSent",
            }
        )
    )


def get_lb_metric_df(
    lb_id: str,
    metric_name: str,
    start_iso: str,
    end_iso: str,
    resolution="1m",
    agg="sum",
):
    fn = {"sum": "sum()", "mean": "mean()", "max": "max()", "min": "min()"}.get(
        agg, "sum()"
    )
    lb_id_escaped = (lb_id or "").replace('"', '\\"')
    query = f'{metric_name}[{resolution}]{{resourceId = "{lb_id_escaped}"}}.{fn}'
    details = oci.monitoring.models.SummarizeMetricsDataDetails(
        namespace=LB_METRICS_NAMESPACE,
        query=query,
        start_time=start_iso,
        end_time=end_iso,
        resolution=resolution,
    )
    resp = mon.summarize_metrics_data(
        compartment_id=COMPARTMENT_ID, summarize_metrics_data_details=details
    )
    rows = []
    for item in resp.data or []:
        for d in item.aggregated_datapoints or []:
            rows.append({"timestamp": d.timestamp, "value": d.value})
    return (
        pd.DataFrame(rows).sort_values("timestamp")
        if rows
        else pd.DataFrame(columns=["timestamp", "value"])
    )


def update_lb_bandwidth(lb_id: str, mbps: int, wait=True):
    details = oci.load_balancer.models.UpdateLoadBalancerShapeDetails(
        minimum_bandwidth_in_mbps=mbps, maximum_bandwidth_in_mbps=mbps
    )
    lbc.update_load_balancer_shape(lb_id, details)
    if wait:
        for _ in range(90):
            lb = lbc.get_load_balancer(lb_id).data
            if getattr(lb, "lifecycle_state", None) == "ACTIVE":
                break
            time.sleep(5)
    print(f"LB cap set to {mbps} Mbps")


def backend_set_health(lb_id: str, backendset_name="backendset1"):
    return lbc.get_backend_set_health(lb_id, backendset_name).data


print("Cell 9 complete: SSH + Monitoring helpers loaded.")
print("NEXT: Run Cell 10 for generator→LB sanity checks.")

In [ ]:
# Cell 10 — Objective: Sanity checks (+ basic connectivity)


def curl_status_from_gen(host, url, timeout=2):
    # HEAD (-I) with TLS verification disabled (-k). Short connect/total timeouts.
    cmd = f"curl -skI --connect-timeout {timeout} --max-time {timeout} {url} | head -n1 || true"
    out, err, rc = ssh_exec(host, command=cmd)
    line = (out or "").strip()
    return ((line if line else "blocked/timeout"), (err or ""), rc)


print("Sanity from each generator to LB (HTTPS):")
for h in GEN_IPS:
    url = f"https://{TARGET_HOST}{HEALTH_ENDPOINT_PATH}"
    st_out, st_err, _ = curl_status_from_gen(h, url, timeout=2)
    print(f"{h} => status: {st_out} | err: {st_err or '(no stderr)'}")

print("Cell 10 complete: Sanity check executed.")
print("NEXT: Run cell 11 for Distributed Locust orchestration.")

In [ ]:
# Cell 11 — Objective: Distributed Locust orchestration (supports CPS & Throughput, payload-aware, auto workers per CPU)
# Uses a small remote run_master.sh to avoid nested-quote issues in tmux.

import os, time, io, paramiko, glob
import pandas as pd
from datetime import datetime, timezone, timedelta

MASTER = GEN_IPS[0] if GEN_IPS else None
WORKERS = GEN_IPS[1:] if GEN_IPS and len(GEN_IPS) > 1 else []
WORKDIR_REMOTE = LOCUST_WORKDIR
RESULTS_DIR_REMOTE = f"{WORKDIR_REMOTE}/results"

TMUX_UI_MASTER = "locust_ui_master"
TMUX_HEADLESS_MASTER = "locust_headless"


def _export_env(mode="cps"):
    mode = (mode or "cps").lower()
    assert mode in ("cps", "throughput")
    return (
        f"export LOCUST_MODE={mode}; "
        f"export LOCUST_HEALTH_PATH='{HEALTH_ENDPOINT_PATH}'; "
        f"export LOCUST_THROUGHPUT_PATH='{THROUGHPUT_ENDPOINT_PATH}'; "
        f"export LOCUST_VERIFY_TLS={'true' if LOCUST_VERIFY_TLS else 'false'}; "
        f"export LOCUST_CONNECT_TIMEOUT_S={LOCUST_CONNECT_TIMEOUT/1000.0}; "
        f"export LOCUST_READ_TIMEOUT_S={LOCUST_READ_TIMEOUT/1000.0}; "
        f"export LOCUST_WAIT_TIME_S={LOCUST_WAIT_TIME_SEC}; "
        f"export PATH=$HOME/.local/bin:/usr/local/bin:/usr/bin:/bin:$PATH; "
    )


def _stop_tmux_session(host, session_name):
    cmd = rf"bash -lc 'tmux has-session -t {session_name} 2>/dev/null && tmux kill-session -t {session_name} || true'"
    ssh_exec(host, command=cmd, timeout=20)


def _ensure_workspace_and_locust(host):
    # Install locust for opc user and verify import
    setup = rf"""bash -lc '
set -e
mkdir -p {RESULTS_DIR_REMOTE}
python3 -m pip install --no-cache-dir --upgrade --user pip || true
python3 -m pip show locust >/dev/null 2>&1 || python3 -m pip install --no-cache-dir --user locust
python3 - <<PY
try:
    import locust
    print("LOCUST_OK", locust.__version__)
except Exception as e:
    print("LOCUST_FAIL", e)
PY
echo READY
'"""
    out, err, rc = ssh_exec(host, command=setup, timeout=240)
    if "LOCUST_OK" not in (out or ""):
        raise RuntimeError(
            f"{host}: locust not available for opc; setup output:\n{out}\n{err}"
        )

    LOCUSTFILE_CODE = r"""import os
from locust import HttpUser, task, constant
VERIFY_TLS        = os.environ.get("LOCUST_VERIFY_TLS", "false").lower() == "true"
CONNECT_TIMEOUT_S = float(os.environ.get("LOCUST_CONNECT_TIMEOUT_S", "8"))
READ_TIMEOUT_S    = float(os.environ.get("LOCUST_READ_TIMEOUT_S", "15"))
WAIT_TIME_S       = float(os.environ.get("LOCUST_WAIT_TIME_S", "1.0"))
MODE              = os.environ.get("LOCUST_MODE", "cps").lower()
HEALTH_PATH       = os.environ.get("LOCUST_HEALTH_PATH", "/healthz")
THROUGHPUT_PATH   = os.environ.get("LOCUST_THROUGHPUT_PATH", "/payload_100k")
class CpsUser(HttpUser):
    wait_time = constant(WAIT_TIME_S)
    @task
    def do_request(self):
        path    = HEALTH_PATH if MODE == "cps" else THROUGHPUT_PATH
        headers = {"Connection": "close"} if MODE == "cps" else {}
        self.client.get(path, headers=headers, verify=VERIFY_TLS,
                        timeout=(CONNECT_TIMEOUT_S, READ_TIMEOUT_S),
                        name=("cps_req" if MODE == "cps" else "throughput_req"))"""

    pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None))
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    try:
        transport = ssh.get_transport()
        sftp = paramiko.SFTPClient.from_transport(transport)
        try:
            sftp.stat(WORKDIR_REMOTE)
        except FileNotFoundError:
            sftp.mkdir(WORKDIR_REMOTE)
        with sftp.file(f"{WORKDIR_REMOTE}/locustfile.py", "w") as f:
            f.set_pipelined(True)
            f.write(LOCUSTFILE_CODE)
        sftp.chmod(f"{WORKDIR_REMOTE}/locustfile.py", 0o644)
        sftp.close()
    finally:
        ssh.close()


def _tmux_session_exists(host, session_name):
    out, err, rc = ssh_exec(
        host,
        command=rf"bash -lc 'tmux has-session -t {session_name} 2>/dev/null && echo YES || echo NO'",
    )
    return (out or "").strip() == "YES"


def _get_private_ip(host):
    if MASTER_PRIVATE_IP_OVERRIDE:
        return MASTER_PRIVATE_IP_OVERRIDE
    cmd = r"""bash -lc '
get_ip() {
  if curl -fsS -H "Authorization: Bearer Oracle" http://169.254.169.254/opc/v2/vnics/ >/tmp/vn.json 2>/dev/null; then :; 
  elif curl -fsS http://169.254.169.254/opc/v1/vnics/ >/tmp/vn.json 2>/dev/null; then :; 
  else : > /tmp/vn.json; fi
  IP=$(tr -d "\n" </tmp/vn.json | sed -n '\''s/.*"privateIp"[[:space:]]*:[[:space:]]*"\([0-9.]*\)".*/\1/p'\'' | head -1)
  [ -n "$IP" ] && echo "$IP" && return 0
  IP=$(ip -4 route get 1.1.1.1 2>/dev/null | awk '\''{for(i=1;i<=NF;i++) if($i=="src") print $(i+1)}'\'' | head -1)
  [ -n "$IP" ] && echo "$IP" && return 0
  IP=$(hostname -I 2>/dev/null | awk '\''{print $1}'\'' | head -1)
  [ -n "$IP" ] && echo "$IP" && return 0
  IP=$(ip -4 addr show scope global 2>/dev/null | awk '\''/inet /{print $2}'\'' | cut -d/ -f1 | head -1)
  [ -n "$IP" ] && echo "$IP" || echo ""
}
get_ip
'"""
    out, err, rc = ssh_exec(host, command=cmd, timeout=20)
    return (out or "").strip()


def _detect_nproc(host):
    cmd = r"bash -lc 'nproc 2>/dev/null || getconf _NPROCESSORS_ONLN 2>/dev/null || echo 1'"
    out, err, rc = ssh_exec(host, command=cmd, timeout=10)
    try:
        n = int((out or "1").strip())
        return max(1, n)
    except Exception:
        return 1


def _count_remote_workers(host):
    cmd = r"""bash -lc "ps -eo cmd | grep -E 'python3 -m locust .*--worker( |$)|locust .*--worker( |$)' | grep -v grep | wc -l" """
    out, err, rc = ssh_exec(host, command=cmd, timeout=10)
    try:
        return int((out or "0").strip())
    except Exception:
        return 0


def _upload_remote_file(host, remote_path: str, text: str, mode: int = 0o755):
    pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None))
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
    try:
        sftp = ssh.open_sftp()
        base_dir = os.path.dirname(remote_path)
        try:
            sftp.stat(base_dir)
        except FileNotFoundError:
            sftp.mkdir(base_dir)
        with sftp.file(remote_path, "w") as f:
            f.write(text)
        sftp.chmod(remote_path, mode)
        sftp.close()
    finally:
        ssh.close()


def _write_and_run_bootstrap(host, script_text):
    # Upload start_workers.sh and run it
    path = f"{WORKDIR_REMOTE}/start_workers.sh"
    _upload_remote_file(host, path, script_text, 0o755)
    # Run in background
    cmd = rf"bash -lc '{_export_env(TEST_MODE)} cd {WORKDIR_REMOTE} && nohup ./start_workers.sh > start_workers.out 2>&1 &'"
    ssh_exec_quick(host, cmd)


def _start_workers(master_host_for_private_ip, mode="cps", include_master=True):
    master_priv = _get_private_ip(master_host_for_private_ip)
    if not master_priv:
        raise RuntimeError(
            "Could not resolve master private IP (override with MASTER_PRIVATE_IP_OVERRIDE in Cell 2)."
        )
    targets = []
    if include_master and MASTER:
        targets.append(MASTER)
    targets.extend(WORKERS or [])
    targets = list(dict.fromkeys([h for h in targets if h]))  # dedupe

    for host in targets:
        _ensure_workspace_and_locust(host)

        if isinstance(WORKERS_PER_HOST, int):
            desired = WORKERS_PER_HOST
        elif isinstance(WORKERS_PER_HOST, str) and WORKERS_PER_HOST.lower() == "auto":
            nproc = _detect_nproc(host)
            desired = nproc - int(CPU_RESERVE)
        else:
            desired = 1
        n = max(int(MIN_WORKERS_PER_HOST), min(int(MAX_WORKERS_PER_HOST), int(desired)))

        bootstrap = f"""#!/usr/bin/env bash
set -euo pipefail
{_export_env(mode)}
cd {WORKDIR_REMOTE}
echo "Spawning {n} worker(s) to master {master_priv} at $(date -u)"
for i in $(seq 1 {n}); do
  nohup python3 -m locust -f locustfile.py --worker --master-host {master_priv} > locust-worker-$i.log 2>&1 &
done
echo "WORKERS_STARTED $(date -u)"
"""
        _write_and_run_bootstrap(host, bootstrap)

        time.sleep(3.0)
        started = _count_remote_workers(host)
        print(
            f"[{host}] master={master_priv} | requested {n} worker(s) | nproc={_detect_nproc(host)} | reserve={CPU_RESERVE} | started≈{started} | mode={mode}"
        )


def _run_master_non_blocking(mode, label, total_users, warmup_sec, hold_sec):
    label_ts = f"{label}_{TS_UTC}"
    users = int(total_users)
    spawn_rate = max(1, int(round(users / max(1, warmup_sec))))
    csv_prefix = f"{RESULTS_DIR_REMOTE}/{label_ts}"
    stats_csv = f"{csv_prefix}_stats.csv"
    html_path = f"{RESULTS_DIR_REMOTE}/{label_ts}.html"
    log_path = f"{RESULTS_DIR_REMOTE}/{label_ts}.log"

    ssh_exec(MASTER, command=f"bash -lc 'mkdir -p {RESULTS_DIR_REMOTE}'", timeout=10)

    _start_workers(MASTER, mode=mode, include_master=True)
    time.sleep(3)

    start_ts = datetime.now(timezone.utc)
    run_time_sec = int(warmup_sec) + int(hold_sec)

    # Upload a small runner script to avoid complex nested quoting for tmux send-keys
    master_runner_path = f"{WORKDIR_REMOTE}/run_master.sh"
    master_runner = f"""#!/usr/bin/env bash
set -euo pipefail
{_export_env(mode)}
cd {WORKDIR_REMOTE}
python3 -m locust -f locustfile.py --master --headless \
  --master-bind-host 0.0.0.0 --host https://{TARGET_HOST} \
  --users {users} --spawn-rate {spawn_rate} --run-time {run_time_sec}s \
  --stop-timeout 30 --only-summary --csv {csv_prefix} --csv-full-history --html {html_path} \
  > {log_path} 2>&1
"""
    _upload_remote_file(MASTER, master_runner_path, master_runner, 0o755)

    # Launch the runner via tmux (simple, no nested quoting)
    master_cmd = rf"bash -lc 'tmux has-session -t {TMUX_HEADLESS_MASTER} 2>/dev/null && tmux kill-session -t {TMUX_HEADLESS_MASTER} || true; tmux new -d -s {TMUX_HEADLESS_MASTER} {master_runner_path}'"
    ssh_exec_quick(MASTER, master_cmd)

    for _ in range(30):
        if _tmux_session_exists(MASTER, TMUX_HEADLESS_MASTER):
            break
        time.sleep(1)

    deadline = time.time() + run_time_sec + GRACE_SEC

    def _remote_file_exists(host, path):
        pkey = _load_pkey(
            SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None)
        )
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
        try:
            sftp = ssh.open_sftp()
            try:
                sftp.stat(path)
                return True
            except FileNotFoundError:
                return False
            finally:
                sftp.close()
        finally:
            ssh.close()

    def _remote_line_count(host, path):
        cmd = rf"bash -lc 'wc -l < {path} 2>/dev/null || echo 0'"
        out, err, rc = ssh_exec(host, command=cmd, timeout=10)
        try:
            return int((out or "0").strip())
        except Exception:
            return 0

    while time.time() < deadline:
        running = _tmux_session_exists(MASTER, TMUX_HEADLESS_MASTER)
        has_csv = _remote_file_exists(MASTER, stats_csv)
        lines = _remote_line_count(MASTER, stats_csv) if has_csv else 0
        if (not running) and has_csv and lines > 1:
            break
        time.sleep(2)

    _stop_tmux_session(MASTER, TMUX_HEADLESS_MASTER)

    end_ts = datetime.now(timezone.utc)
    print(
        f"[MASTER] mode={mode} label={label_ts} users={users} spawn_rate={spawn_rate}/s | run_time={run_time_sec}s | absolute_window≈{run_time_sec+GRACE_SEC}s"
    )
    return start_ts, end_ts, label_ts


def _collect_results_to_local(master_ip, label_for_files, local_dir):
    os.makedirs(local_dir, exist_ok=True)
    pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None))
    transport = paramiko.Transport((master_ip, 22))
    transport.connect(username="opc", pkey=pkey)
    sftp = paramiko.SFTPClient.from_transport(transport)
    base = RESULTS_DIR_REMOTE
    files = [
        f"{label_for_files}_stats.csv",
        f"{label_for_files}_stats_history.csv",
        f"{label_for_files}_failures.csv",
        f"{label_for_files}_exceptions.csv",
        f"{label_for_files}.html",
        f"{label_for_files}.log",
    ]
    for fn in files:
        remote = f"{base}/{fn}"
        try:
            sftp.get(remote, os.path.join(local_dir, fn))
            print("Downloaded:", fn)
        except Exception as e:
            print("Skip missing:", fn, e)
    sftp.close()
    transport.close()


def run_cps_locust_once(total_cps: int, warmup_sec: int, hold_sec: int, label: str):
    if not MASTER:
        print("No generator hosts available.")
        return None
    for h in GEN_IPS:
        _ensure_workspace_and_locust(h)
    print(
        f"[CPS] {label} | total={total_cps} | warmup={warmup_sec}s | hold={hold_sec}s | users={total_cps} | spawn_rate≈{max(1,int(round(total_cps/max(1,warmup_sec))))}/s"
    )
    start_ts, end_ts, label_ts = _run_master_non_blocking(
        "cps", label, total_cps, warmup_sec, hold_sec
    )
    local_result_dir = os.path.join(OUTPUT_DIR, f"locust_{label_ts}")
    _collect_results_to_local(MASTER, label_ts, local_result_dir)
    return {
        "label": label_ts,
        "start_ts": start_ts,
        "end_ts": end_ts,
        "local_dir": local_result_dir,
    }


def run_throughput_locust_once(
    target_gbps: float, warmup_sec: int, hold_sec: int, label_prefix: str
):
    if not MASTER:
        print("No generator hosts available.")
        return None
    for h in GEN_IPS:
        _ensure_workspace_and_locust(h)
    bytes_per_req = max(1, int(TPUT_PAYLOAD_BYTES_PER_REQ))
    rps = int(round(float(target_gbps) * 1e9 / 8.0 / bytes_per_req))
    label = f"{label_prefix}_{str(target_gbps).replace('.', '_')}gbps_{hold_sec}s"
    print(
        f"[THROUGHPUT] {label} | payload={TPUT_PAYLOAD_SIZE_LABEL} (~{bytes_per_req} bytes) | target_gbps={target_gbps} ⇒ rps≈{rps} | warmup={warmup_sec}s | hold={hold_sec}s | users={rps} | spawn_rate≈{max(1,int(round(rps/max(1,warmup_sec))))}/s"
    )
    start_ts, end_ts, label_ts = _run_master_non_blocking(
        "throughput", label, rps, warmup_sec, hold_sec
    )
    local_result_dir = os.path.join(OUTPUT_DIR, f"locust_{label_ts}")
    _collect_results_to_local(MASTER, label_ts, local_result_dir)
    return {
        "label": label_ts,
        "start_ts": start_ts,
        "end_ts": end_ts,
        "local_dir": local_result_dir,
    }


def inspect_state():
    if MASTER:
        out, err, rc = ssh_exec(
            MASTER, command="bash -lc 'tmux ls 2>/dev/null || true'"
        )
        print(f"[MASTER tmux]\n{out or '(no sessions)'}")
    for w in WORKERS or []:
        out, err, rc = ssh_exec(w, command="bash -lc 'tmux ls 2>/dev/null || true'")
        print(f"[{w} tmux]\n{out or '(no sessions)'}")


def stop_all_locust():
    # This only stops tmux-controlled masters. Workers are nohup'ed; use pkill if you want a hard stop.
    if MASTER:
        _stop_tmux_session(MASTER, TMUX_HEADLESS_MASTER)
        _stop_tmux_session(MASTER, TMUX_UI_MASTER)
    hosts = WORKERS or []
    for h in hosts:
        _stop_tmux_session(h, "locust_workers")
    print(
        "stop_all_locust: tmux sessions stopped. Worker processes continue (nohup). Use pkill if a hard stop is required."
    )


print(
    "Cell 11 complete: Orchestration loaded (workers via nohup bootstrap; auto CPU; per-worker logs)."
)
print("NEXT: Run Cell 12 to start the UI; verify workers in UI (should be > 0).")

In [ ]:
# Cell 12 — Objective: Start Locust UI master + workers (uses selected Test Mode + payload)
# Launches UI via a small remote run_ui.sh (no nested quoting); then starts workers on all generators.

import time
import paramiko


def _ensure_upload_helper():
    # Define _upload_remote_file if not already available (keeps notebook self-contained)
    try:
        _upload_remote_file  # noqa
        return
    except NameError:
        pass

    def _upload_remote_file(host, remote_path: str, text: str, mode: int = 0o755):
        pkey = _load_pkey(
            SSH_PRIVATE_KEY_PATH, os.environ.get("SSH_KEY_PASSPHRASE", None)
        )
        ssh = paramiko.SSHClient()
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        ssh.connect(hostname=host, username="opc", pkey=pkey, timeout=30)
        try:
            sftp = ssh.open_sftp()
            base_dir = os.path.dirname(remote_path)
            try:
                sftp.stat(base_dir)
            except FileNotFoundError:
                sftp.mkdir(base_dir)
            with sftp.file(remote_path, "w") as f:
                f.write(text)
            sftp.chmod(remote_path, mode)
            sftp.close()
        finally:
            ssh.close()

    globals()["_upload_remote_file"] = _upload_remote_file


def start_ui_master_and_workers(mode=None):
    mode = (mode or TEST_MODE or "cps").lower()
    if not MASTER:
        print("No generator hosts available.")
        return None
    for h in GEN_IPS:
        _ensure_workspace_and_locust(h)

    _ensure_upload_helper()

    # Prepare a simple UI runner script (no nested quoting)
    ui_runner_path = f"{WORKDIR_REMOTE}/run_ui.sh"
    ui_runner = f"""#!/usr/bin/env bash
set -euo pipefail
{_export_env(mode)}
cd {WORKDIR_REMOTE}
python3 -m locust -f locustfile.py --master --master-bind-host 0.0.0.0 --web-host {UI_WEB_HOST} --web-port {UI_WEB_PORT}
"""
    _upload_remote_file(MASTER, ui_runner_path, ui_runner, 0o755)

    # Launch UI master via tmux
    _stop_tmux_session(MASTER, TMUX_UI_MASTER)
    cmd_master = rf"bash -lc 'tmux has-session -t {TMUX_UI_MASTER} 2>/dev/null && tmux kill-session -t {TMUX_UI_MASTER} || true; tmux new -d -s {TMUX_UI_MASTER} {ui_runner_path}'"
    ssh_exec_quick(MASTER, cmd_master)
    print(
        f"[MASTER] UI start issued (tmux) with mode={mode} and throughput_path={THROUGHPUT_ENDPOINT_PATH}."
    )

    # Start workers on all generators INCLUDING master
    _start_workers(MASTER, mode=mode, include_master=True)

    time.sleep(3)
    print("\n[STATE] Master/workers after launch:")
    inspect_state()

    print("\nUI access options (SSH tunnel recommended):")
    print(
        f"  ssh -i {SSH_PRIVATE_KEY_PATH} -o IdentitiesOnly=yes -N -L 8089:localhost:{UI_WEB_PORT} opc@{MASTER}"
    )
    print("  Then open: http://localhost:8089")
    print("If 8089 is busy locally, use 8088:")
    print(
        f"  ssh -i {SSH_PRIVATE_KEY_PATH} -o IdentitiesOnly=yes -N -L 8088:localhost:{UI_WEB_PORT} opc@{MASTER}"
    )
    print("  Then open: http://localhost:8088")

    host_url = f"https://{TARGET_HOST}" if TARGET_HOST else "(unknown — run Cell 8)"
    print("\nUI form values (copy/paste):")
    print(f"  Host: {host_url}")
    print(f"  Health path: {HEALTH_ENDPOINT_PATH}")
    print(
        f"  Throughput path: {THROUGHPUT_ENDPOINT_PATH}  (payload={TPUT_PAYLOAD_SIZE_LABEL} ~{TPUT_PAYLOAD_SIZE_BYTES} bytes)"
    )
    print(f"  TLS verify (locustfile): {LOCUST_VERIFY_TLS}")
    print(
        f"  Timeouts (seconds): connect={LOCUST_CONNECT_TIMEOUT/1000.0:.2f} read={LOCUST_READ_TIMEOUT/1000.0:.2f}"
    )

    def _sr(users, warm):
        warm = max(1, int(warm))
        return max(1, int(round(users / warm)))

    if TEST_MODE == "cps":
        print("\nRecommended UI inputs per CPS tier (users, spawn_rate):")
        for tier in [10000, 25000, 35000, 50000, 100000]:
            w = int(CPS_WARMUPS.get(tier, 120))
            print(
                f"  Tier {tier}: users={tier}, spawn_rate={_sr(tier, w)}/s (warmup={w}s)"
            )
    else:
        print("\nThroughput mode: examples for selected payload:")
        bpr = max(1, int(TPUT_PAYLOAD_BYTES_PER_REQ))
        rps_1 = int(round(1e9 / 8 / bpr))
        rps_5 = rps_1 * 5
        rps_10 = rps_1 * 10
        print(
            f"  Payload={TPUT_PAYLOAD_SIZE_LABEL} (~{bpr} bytes) ⇒ RPS targets: 1 Gbps≈{rps_1}, 5 Gbps≈{rps_5}, 10 Gbps≈{rps_10}"
        )
        print(
            "  Pick 'users' equal to desired RPS; spawn_rate ~ users / warmup_seconds"
        )


# Launch immediately using current TEST_MODE
start_ui_master_and_workers(TEST_MODE)

print(
    "Cell 12 complete: UI master/workers launched (python3 -m locust; PATH exported)."
)
print(
    "NEXT: If you want UI exposure via NSG, use Cell 13 open_ui_port() (stateless). Otherwise proceed to warm-up (Cell 16) and suite (Cell 17)."
)

In [ ]:
# Cell X — Objective: Robust backend conntrack/backlog probe via bastion (no NSG edits, upload + run script)

import os, json, time
import paramiko

# Inputs from earlier cells
GEN_IPS = globals().get("GEN_IPS", [])
BACKEND_IPS = globals().get("BACKEND_IPS", [])
MASTER = GEN_IPS[0] if GEN_IPS else None
SSH_PRIVATE_KEY_PATH = os.path.expanduser(
    globals().get(
        "SSH_PRIVATE_KEY_PATH", os.environ.get("SSH_PRIVATE_KEY_PATH", "~/.ssh/id_rsa")
    )
)
SSH_KEY_PASSPHRASE = os.environ.get("SSH_KEY_PASSPHRASE", None)
OUTPUT_DIR = globals().get("OUTPUT_DIR", "./results")

assert MASTER, "MASTER generator IP not set (run Cell 8)."
assert BACKEND_IPS, "BACKEND_IPS not set (run Cell 8)."
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASTION_USER = "opc"
TARGET_USER = "opc"
SSH_TIMEOUT_S = 30
REMOTE_SCRIPT = "/tmp/cps_probe.sh"

# Small probe script (no fancy quoting). Reads-only; no sudo required for typical OL images.
PROBE_SCRIPT = """#!/usr/bin/env bash
set -euo pipefail
read_sys() {
  local p="$1"; local k="$2"; local v
  if [ -r "$p" ]; then v=$(cat "$p" 2>/dev/null || true); else v=$(sysctl -n "$k" 2>/dev/null || echo -1); fi
  echo "$v"
}
COUNT=$(read_sys /proc/sys/net/netfilter/nf_conntrack_count net.netfilter.nf_conntrack_count)
MAX=$(read_sys /proc/sys/net/netfilter/nf_conntrack_max   net.netfilter.nf_conntrack_max)
HASH=$(cat /sys/module/nf_conntrack/parameters/hashsize 2>/dev/null || echo -1)
ABORT=$(sysctl -n net.ipv4.tcp_abort_on_overflow 2>/dev/null || echo -1)
SOMAX=$(sysctl -n net.core.somaxconn 2>/dev/null || echo -1)
SYNBK=$(sysctl -n net.ipv4.tcp_max_syn_backlog 2>/dev/null || echo -1)
NDBK=$(sysctl -n net.core.netdev_max_backlog 2>/dev/null || echo -1)
PORTS=$(sysctl -n net.ipv4.ip_local_port_range 2>/dev/null || echo "")
UNAME=$(uname -r 2>/dev/null || echo "")
UPTIME=$(uptime 2>/dev/null || echo "")
NPROC=$( (nproc 2>/dev/null || getconf _NPROCESSORS_ONLN 2>/dev/null) || echo 0 )
# ss listing (avoid nested quotes); fall back to full listing
SS80=$(ss -ltn 2>/dev/null | grep -E "LISTEN" | grep -E ":80( |$)" || true)
# nginx listen lines
NGXL=$(grep -n "listen 80" /etc/nginx/conf.d/*.conf 2>/dev/null || true)
# Emit simple key=value lines
pct=""
if [ "$COUNT" != "-1" ] && [ "$MAX" != "-1" ] && [ "$MAX" != "0" ]; then
  pct=$(awk -v c="$COUNT" -v m="$MAX" 'BEGIN{printf "%.2f", (100.0*c/m)}')
fi
cat <<OUT
kernel=$UNAME
uptime=$UPTIME
nproc=$NPROC
nf_conntrack_count=$COUNT
nf_conntrack_max=$MAX
nf_conntrack_hashsize=$HASH
conntrack_usage_pct=$pct
tcp_abort_on_overflow=$ABORT
somaxconn=$SOMAX
tcp_max_syn_backlog=$SYNBK
netdev_max_backlog=$NDBK
ip_local_port_range=$PORTS
ss_listen_80=$SS80
nginx_listen_lines=$NGXL
OUT
"""

# SSH helpers (bastion hop + SFTP over the tunneled session)


def _load_pkey(path, passphrase=None):
    excs = []
    for Key in (paramiko.RSAKey, paramiko.ECDSAKey, paramiko.Ed25519Key):
        try:
            return Key.from_private_key_file(path, password=passphrase)
        except Exception as e:
            excs.append(str(e))
    raise RuntimeError(f"Could not load SSH key {path}: {'; '.join(excs)}")


pkey = _load_pkey(SSH_PRIVATE_KEY_PATH, SSH_KEY_PASSPHRASE)


def _connect_bastion(host):
    cli = paramiko.SSHClient()
    cli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    cli.connect(hostname=host, username=BASTION_USER, pkey=pkey, timeout=SSH_TIMEOUT_S)
    return cli


def _connect_target_via(bastion_cli, target_host):
    transport = bastion_cli.get_transport()
    chan = transport.open_channel("direct-tcpip", (target_host, 22), ("", 0))
    tcli = paramiko.SSHClient()
    tcli.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    tcli.connect(
        hostname=target_host,
        username=TARGET_USER,
        pkey=pkey,
        sock=chan,
        timeout=SSH_TIMEOUT_S,
    )
    return tcli


def _upload_and_run(bastion_cli, target_host, script_text):
    tcli = _connect_target_via(bastion_cli, target_host)
    try:
        sftp = tcli.open_sftp()
        try:
            with sftp.file(REMOTE_SCRIPT, "w") as f:
                f.write(script_text)
            sftp.chmod(REMOTE_SCRIPT, 0o755)
        finally:
            sftp.close()
        stdin, stdout, stderr = tcli.exec_command(
            f"bash {REMOTE_SCRIPT}", timeout=SSH_TIMEOUT_S
        )
        out = stdout.read().decode("utf-8", "ignore")
        err = stderr.read().decode("utf-8", "ignore")
        rc = stdout.channel.recv_exit_status()
        return rc, out, err
    finally:
        tcli.close()


# Run across all backends
results = {}
with _connect_bastion(MASTER) as bcli:
    for ip in BACKEND_IPS:
        try:
            rc, out, err = _upload_and_run(bcli, ip, PROBE_SCRIPT)
            if rc != 0:
                results[ip] = {"error": f"rc={rc}", "stderr": err, "stdout": out}
                continue
            data = {}
            for line in (out or "").splitlines():
                if "=" in line:
                    k, v = line.split("=", 1)
                    data[k.strip()] = v.strip()
            # normalize pct
            try:
                if (
                    not data.get("conntrack_usage_pct")
                    and data.get("nf_conntrack_count")
                    and data.get("nf_conntrack_max")
                ):
                    c = float(data["nf_conntrack_count"])
                    m = float(data["nf_conntrack_max"])
                    data["conntrack_usage_pct"] = f"{(100.0*c/m):.2f}"
            except Exception:
                pass
            results[ip] = data
        except Exception as e:
            results[ip] = {"exception": str(e)}

print("\nBackend conntrack/backlog summary:")
for ip, info in results.items():
    print(f"\n[{ip}]")
    if "exception" in info or "error" in info:
        print(json.dumps(info, indent=2))
        continue
    fields = [
        "kernel",
        "uptime",
        "nproc",
        "nf_conntrack_count",
        "nf_conntrack_max",
        "nf_conntrack_hashsize",
        "conntrack_usage_pct",
        "tcp_abort_on_overflow",
        "somaxconn",
        "tcp_max_syn_backlog",
        "netdev_max_backlog",
        "ip_local_port_range",
    ]
    for f in fields:
        print(f"  {f}: {info.get(f)}")
    print("  ss_listen_80:", info.get("ss_listen_80", ""))
    print("  nginx_listen_lines:", info.get("nginx_listen_lines", ""))

# Save JSON
from datetime import datetime, timezone

COLLECT_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
json_path = os.path.join(OUTPUT_DIR, f"backend_conntrack_probe_{COLLECT_TS}.json")
with open(json_path, "w") as f:
    json.dump({"backends": results, "master": MASTER}, f, indent=2)
print("\nSaved JSON:", json_path)

In [ ]:
# Cell 13 — Objective: Optional — Open/close NSG for UI port (stateless, port/CIDR aware)

import oci


def _vn_client():
    return oci.core.VirtualNetworkClient(
        oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
    )


def _get_nsg_by_name(compartment_id, name):
    net = _vn_client()
    nsgs = oci.pagination.list_call_get_all_results(
        net.list_network_security_groups,
        compartment_id=compartment_id,
        display_name=name,
    ).data
    return nsgs[0] if nsgs else None


def _list_rules(nsg_id):
    net = _vn_client()
    return net.list_network_security_group_security_rules(nsg_id).data


def _rule_matches_port_and_cidr(rule, port, cidr_or_none):
    try:
        if rule.direction != "INGRESS":
            return False
        if rule.protocol not in ("6", "all"):  # 6 = TCP
            return False
        dpr = (
            rule.tcp_options.destination_port_range
            if getattr(rule, "tcp_options", None)
            else None
        )
        if not dpr or dpr.min != port or dpr.max != port:
            return False
        if cidr_or_none is None:
            return True
        return (rule.source_type == "CIDR_BLOCK") and (rule.source == cidr_or_none)
    except Exception:
        return False


def open_ui_port():
    """
    Idempotently open TCP/UI_WEB_PORT on nsg-generators from UI_ALLOWED_CIDR as STATELESS.
    Safe to run multiple times. No effect if already present.
    """
    if UI_EXPOSE_MODE != "nsg":
        print(
            f"UI_EXPOSE_MODE={UI_EXPOSE_MODE} (no NSG change needed). Use SSH tunnel instead."
        )
        return

    if not COMPARTMENT_ID:
        print("COMPARTMENT_ID is empty; run Cells 8–10 first.")
        return

    nsg = _get_nsg_by_name(COMPARTMENT_ID, "nsg-generators")
    if not nsg:
        print("nsg-generators not found in this compartment.")
        return

    rules = _list_rules(nsg.id)
    already = [
        r for r in rules if _rule_matches_port_and_cidr(r, UI_WEB_PORT, UI_ALLOWED_CIDR)
    ]
    if already:
        print(
            f"NSG already allows TCP/{UI_WEB_PORT} from {UI_ALLOWED_CIDR} on nsg-generators ({nsg.id})"
        )
        return

    net = _vn_client()
    add_spec = oci.core.models.AddNetworkSecurityGroupSecurityRulesDetails(
        security_rules=[
            oci.core.models.AddSecurityRuleDetails(
                direction="INGRESS",
                protocol="6",
                source_type="CIDR_BLOCK",
                source=UI_ALLOWED_CIDR,
                is_stateless=True,  # STATELESS for consistency with stateless posture
                tcp_options=oci.core.models.TcpOptions(
                    destination_port_range=oci.core.models.PortRange(
                        min=UI_WEB_PORT,
                        max=UI_WEB_PORT,
                    )
                ),
                description=f"Locust UI TCP/{UI_WEB_PORT} (stateless)",
            )
        ]
    )
    net.add_network_security_group_security_rules(nsg.id, add_spec)
    print(
        f"NSG rule added (stateless): allow TCP/{UI_WEB_PORT} from {UI_ALLOWED_CIDR} on nsg-generators ({nsg.id})"
    )


def close_ui_port(remove_all_sources=True, source_cidr=None):
    """
    Remove UI rules. By default removes any rule for TCP/UI_WEB_PORT regardless of source.
    Set remove_all_sources=False and source_cidr=<cidr> to remove a specific CIDR only.
    """
    if not COMPARTMENT_ID:
        print("COMPARTMENT_ID is empty; run Cells 8–10 first.")
        return

    nsg = _get_nsg_by_name(COMPARTMENT_ID, "nsg-generators")
    if not nsg:
        print("nsg-generators not found.")
        return

    rules = _list_rules(nsg.id)
    cidr = None if remove_all_sources else (source_cidr or UI_ALLOWED_CIDR)
    targets = [r.id for r in rules if _rule_matches_port_and_cidr(r, UI_WEB_PORT, cidr)]

    if not targets:
        print(
            f"No matching UI rules found for TCP/{UI_WEB_PORT} (cidr={'any' if cidr is None else cidr})."
        )
        return

    net = _vn_client()
    net.remove_network_security_group_security_rules(
        nsg.id,
        oci.core.models.RemoveNetworkSecurityGroupSecurityRulesDetails(
            security_rule_ids=targets
        ),
    )
    print(
        f"Removed {len(targets)} rule(s) for TCP/{UI_WEB_PORT} (cidr={'any' if cidr is None else cidr})."
    )


print("Cell 13 complete: NSG UI helpers ready (stateless).")
print("Usage:")
print(
    "  open_ui_port()   # open UI_WEB_PORT from UI_ALLOWED_CIDR on nsg-generators (stateless; if UI_EXPOSE_MODE=='nsg')"
)
print(
    "  close_ui_port()  # remove all rules for UI_WEB_PORT (or set remove_all_sources=False, source_cidr=...)"
)

In [ ]:
# Cell 14 — Objective: UI API helpers (start/stop via API; preview stats; reset; health)
# Supports calling the Locust UI either via:
#  - direct master IP (MASTER public IP): where="master"
#  - local SSH tunnel to UI_WEB_PORT:    where="local"
#
# Example usage:
#   ui_health()                      # ping UI via master IP
#   ui_health(where="local")         # ping UI via local tunnel
#   ui_swarm(1000, 10, host=f"https://{TARGET_HOST}", where="local")
#   ui_stats_preview(where="local")
#   ui_reset(where="local")
#   ui_stop(where="local")

import os
import requests
import json

REQUEST_TIMEOUT_S = 5  # small API timeout; adjust if needed


def ui_base_url(where: str = "master") -> str:
    """
    Return the base URL for the Locust UI API.
    where = "master" uses MASTER public IP (remote service).
    where = "local"  uses localhost (assumes SSH tunnel to UI_WEB_PORT).
    """
    where = (where or "master").lower()
    if where == "local":
        return f"http://localhost:{UI_WEB_PORT}"
    return f"http://{MASTER}:{UI_WEB_PORT}"


def compute_spawn_rate(users: int, warmup_sec: int) -> int:
    warm = max(1, int(warmup_sec))
    return max(1, int(round(int(users) / warm)))


def ui_health(where: str = "master"):
    url = ui_base_url(where)
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT_S)
        print("ui_health:", r.status_code, (r.text or "")[:120].replace("\n", " "))
    except Exception as e:
        print("ui_health error:", e)


def ui_swarm(
    users: int, spawn_rate: float | int, host: str | None = None, where: str = "master"
):
    """
    Start swarming with user_count and spawn_rate.
    Optional: set 'host' to override the UI form host (e.g., f"https://{TARGET_HOST}").
    """
    base = ui_base_url(where)
    data = {"user_count": int(users), "spawn_rate": float(spawn_rate)}
    if host:
        data["host"] = str(host)
    try:
        r = requests.post(f"{base}/swarm", data=data, timeout=REQUEST_TIMEOUT_S)
        print("ui_swarm status:", r.status_code)
        if r.text:
            print((r.text[:500]))
    except Exception as e:
        print("ui_swarm error:", e)


def ui_stop(where: str = "master"):
    base = ui_base_url(where)
    try:
        r = requests.get(f"{base}/stop", timeout=REQUEST_TIMEOUT_S)
        print("ui_stop status:", r.status_code, (r.text or "")[:200])
    except Exception as e:
        print("ui_stop error:", e)


def ui_reset(where: str = "master"):
    """
    Reset all statistics (clears history counters). Useful between runs.
    """
    base = ui_base_url(where)
    try:
        r = requests.get(f"{base}/stats/reset", timeout=REQUEST_TIMEOUT_S)
        print("ui_reset status:", r.status_code, (r.text or "")[:200])
    except Exception as e:
        print("ui_reset error:", e)


def ui_stats_preview(where: str = "master", preview_bytes: int = 600):
    """
    Print a short preview of /stats/requests JSON.
    """
    base = ui_base_url(where)
    try:
        r = requests.get(f"{base}/stats/requests", timeout=REQUEST_TIMEOUT_S)
        print("ui_stats status:", r.status_code)
        text = r.text or ""
        print(text[:preview_bytes])
    except Exception as e:
        print("ui_stats error:", e)


print("Cell 14 complete: UI API helpers ready (health, swarm, stop, reset, preview).")
print("Examples:")
print(f"  ui_health()                       # via master IP")
print(f"  ui_health(where='local')          # via local SSH tunnel")
print(f"  ui_swarm(1000, 10, host='https://{TARGET_HOST}', where='local')")
print(f"  ui_stats_preview(where='local')")
print(f"  ui_reset(where='local')")
print(f"  ui_stop(where='local')")

In [ ]:
# Cell 15 — Objective: Stop Locust UI and workers (graceful first; hard stop optional)

import time

# Patterns used by pkill to terminate worker processes
_WORKER_PATTERNS = [r"python3 -m locust .*--worker( |$)", r"locust .*--worker( |$)"]


def _pkill_workers(host):
    """
    Issue pkill for worker processes on a single host. Returns (stdout, stderr, rc).
    """
    # Single compound command to try both patterns without failing early
    cmd = r"""bash -lc '
set +e
pkill -f "python3 -m locust .*--worker" 2>/dev/null || true
pkill -f "locust .*--worker" 2>/dev/null || true
# Return success regardless; pkill returns non-zero if no matches
true
'"""
    return ssh_exec(host, command=cmd, timeout=10)


def _count_workers(host):
    """
    Count worker processes on host for visibility.
    """
    cmd = r"""bash -lc "ps -eo cmd | grep -E 'python3 -m locust .*--worker( |$)|locust .*--worker( |$)' | grep -v grep | wc -l" """
    out, err, rc = ssh_exec(host, command=cmd, timeout=10)
    try:
        return int((out or "0").strip())
    except Exception:
        return 0


def stop_ui_master(where: str = "master"):
    """
    Gracefully stop UI swarm via API, then kill the UI tmux session.
    where = "master" hits http://MASTER:UI_WEB_PORT
    where = "local"  hits http://localhost:UI_WEB_PORT (SSH tunnel)
    """
    try:
        ui_stop(where=where)  # best-effort API stop
    except Exception:
        pass
    # Kill UI tmux session for certainty
    _stop_tmux_session(MASTER, TMUX_UI_MASTER)
    time.sleep(0.5)
    print("UI master stopped (API attempted, tmux session terminated).")


def stop_headless_master():
    """
    Kill any headless master tmux session (created by headless helper in earlier cells).
    """
    _stop_tmux_session(MASTER, TMUX_HEADLESS_MASTER)
    time.sleep(0.5)
    print("Headless master stopped (tmux session terminated).")


def stop_all_workers_hard(include_master: bool = True, wait_sec: int = 2):
    """
    Hard stop worker processes (nohup-launched) across master + all workers.
    """
    hosts = ([MASTER] if (include_master and MASTER) else []) + (WORKERS or [])
    if not hosts:
        print("No generator hosts to stop.")
        return
    before = {}
    for h in hosts:
        before[h] = _count_workers(h)
    for h in hosts:
        out, err, rc = _pkill_workers(h)
        print(f"[{h}] pkill issued (workers before={before[h]})")
    time.sleep(wait_sec)
    after = {}
    for h in hosts:
        after[h] = _count_workers(h)
        print(f"[{h}] workers remaining≈{after[h]}")
    print("Worker hard-stop sweep complete.")


def stop_everything(where: str = "master", include_master_workers: bool = True):
    """
    One-call orchestrated stop:
      1) UI API stop (where), then tmux UI cleanup
      2) Kill headless master tmux
      3) pkill workers (master + all workers)
    """
    print(f"Stopping UI (where={where}) ...")
    stop_ui_master(where=where)
    print("Stopping headless master ...")
    stop_headless_master()
    print("Stopping workers (hard kill) ...")
    stop_all_workers_hard(include_master=include_master_workers)
    print("All Locust components stopped.")


def cleanup_worker_logs(hosts: list[str] | None = None, remove: bool = False):
    """
    Show (or delete if remove=True) nohup/log files under WORKDIR_REMOTE on generator hosts.
    """
    targets = hosts or ([MASTER] if MASTER else []) + (WORKERS or [])
    if not targets:
        print("No hosts to inspect.")
        return
    action = "rm -f" if remove else "ls -l"
    for h in targets:
        cmd = rf"""bash -lc 'cd {WORKDIR_REMOTE} 2>/dev/null || exit 0; {action} locust-worker-*.log start_workers.out 2>/dev/null || true'"""
        out, err, rc = ssh_exec(h, command=cmd, timeout=10)
        hdr = "Removed:" if remove else "Logs:"
        if (out or "").strip():
            print(f"[{h}] {hdr}\n{out.strip()}")
        else:
            print(f"[{h}] {hdr} (none)")


print("Cell 15 complete: stop helpers ready.")
print("Examples:")
print(
    "  stop_ui_master(where='local')    # if using SSH tunnel; or where='master' for direct"
)
print("  stop_headless_master()")
print("  stop_all_workers_hard()")
print("  stop_everything(where='local')   # stop UI, headless master, and workers")
print(
    "  cleanup_worker_logs(remove=False) # list worker logs; set remove=True to delete"
)
stop_everything()

In [ ]:
# Cell 16 — Objective: Warm-up run based on Test Mode (CPS or Throughput, payload-aware)
# - Ensures a clean slate (graceful UI stop + tmux cleanup; falls back to legacy stop)
# - Executes a small warm-up for the selected TEST_MODE
# - Uses Cell 11 headless helpers: run_cps_locust_once / run_throughput_locust_once

import os
import glob
import pandas as pd

# Try to stop any previous sessions (prefer new helpers from Cell 15; else fallback)
try:
    # Attempt graceful UI API stop via local tunnel (if present), then tmux cleanup
    stop_ui_master(where="local")
    stop_headless_master()
except NameError:
    try:
        stop_all_locust()
    except NameError:
        print("Stop helpers not available yet; continuing.")

# Optional: clear lingering worker processes/logs (uncomment if you want a hard reset)
# try:
#     stop_all_workers_hard(include_master=True)
#     cleanup_worker_logs(remove=True)
# except NameError:
#     pass

if TEST_MODE == "cps":
    # Warm-up at 1k CPS: spawn_rate ≈ users / warmup_seconds
    warm_seconds = int(CPS_WARMUPS.get(1000, 60))
    hold_seconds = 60
    label = f"warmup_cps_1k_{hold_seconds}s"
    print(f"[Warm-up CPS] users=1000 | warmup={warm_seconds}s | hold={hold_seconds}s")
    warmup = run_cps_locust_once(
        1000, warmup_sec=warm_seconds, hold_sec=hold_seconds, label=label
    )
    print("Warm-up CPS done:", warmup)

else:
    # Throughput warm-up at the first configured Gbps target
    try:
        targets = [
            float(x.strip())
            for x in (TPUT_TARGETS_GBPS_TEXT or "1").split(",")
            if x.strip()
        ]
    except Exception:
        targets = [1.0]
    gbps = targets[0] if targets else 1.0
    print(
        f"[Warm-up Throughput] target_gbps={gbps} | payload={TPUT_PAYLOAD_SIZE_LABEL} (~{TPUT_PAYLOAD_SIZE_BYTES} bytes)"
    )
    warmup = run_throughput_locust_once(
        target_gbps=gbps,
        warmup_sec=TPUT_WARMUP_SEC,
        hold_sec=TPUT_HOLD_SEC,
        label_prefix="tput_warmup",
    )
    print("Warm-up Throughput done:", warmup)

print("Cell 16 complete: Warm-up executed.")
print(
    "NEXT: Run Cell 17 to execute the full suite for the selected mode, then Cell 18 for LB metrics (CPS/throughput)."
)

In [ ]:
# Cell 17 — Objective: Full Suite based on Test Mode (CPS tiers or Throughput targets)

SUITE_RESULTS = []

if TEST_MODE == "cps":
    for tier in [10000, 25000, 35000, 50000, 100000]:
        w = int(CPS_WARMUPS.get(tier, 120))
        h = int(CPS_HOLDS.get(tier, 600))
        base_label = f"cps_{tier}_{h}s"
        res = run_cps_locust_once(tier, warmup_sec=w, hold_sec=h, label=base_label)
        SUITE_RESULTS.append(res)
    print("CPS suite complete.")
else:
    try:
        targets = [
            float(x.strip())
            for x in (TPUT_TARGETS_GBPS_TEXT or "").split(",")
            if x.strip()
        ]
    except Exception:
        targets = [1.0, 5.0, 10.0]
    for gbps in targets or [1.0, 5.0, 10.0]:
        res = run_throughput_locust_once(
            target_gbps=float(gbps),
            warmup_sec=TPUT_WARMUP_SEC,
            hold_sec=TPUT_HOLD_SEC,
            label_prefix="tput",
        )
        SUITE_RESULTS.append(res)
    print("Throughput suite complete.")

print("Cell 17 complete: Suite finished based on Test Mode.")
print("NEXT: Run Cell 18 to pull OCI LB metrics per run and compute avg/peak Gbps.")

In [ ]:
# Cell 18 — Objective: Connection + Throughput metrics (compute average/peak Gbps from BytesSent per run)

import os, glob, time, json
import oci
import pandas as pd
from datetime import datetime, timezone, timedelta


def _iso_utc(dt):
    return dt.replace(tzinfo=timezone.utc).isoformat().replace("+00:00", "Z")


AVAILABLE = set(list_available_lb_metrics(LB_ID) or [])
ALIASES = {
    "CPS": ["NewConnections", "AcceptedConnections", "HandledConnections"],
    "Active": ["ActiveConnections"],
    "Closed": ["ClosedConnections"],
    "BytesIn": ["BytesIn", "BytesReceived"],
    "BytesOut": ["BytesOut", "BytesSent"],
}


def pick_metric(key):
    for name in ALIASES.get(key, []):
        if name in AVAILABLE:
            return name
    return ALIASES.get(key, [None])[0]


def _newest_artifact_utc(local_dir: str):
    paths = []
    for pat in ("*_stats.csv", "*_stats_history.csv", "*.html", "*.log"):
        paths.extend(glob.glob(os.path.join(local_dir, pat)))
    if not paths:
        return None
    mt = max(os.path.getmtime(p) for p in paths if os.path.exists(p))
    return datetime.fromtimestamp(mt, tz=timezone.utc)


def _parse_label_duration_seconds(label: str):
    try:
        parts = label.split("_")
        hold_s = int(parts[2].rstrip("s"))
        if parts[0] == "cps":
            tier = int(parts[1])
            warm_s = int(CPS_WARMUPS.get(tier, 120))
        else:
            warm_s = int(TPUT_WARMUP_SEC)
        return warm_s + hold_s
    except Exception:
        return None


def _derive_window_from_artifacts(item):
    local_dir = item.get("local_dir")
    if not local_dir or not os.path.isdir(local_dir):
        return None, None
    end_guess = _newest_artifact_utc(local_dir)
    hist = glob.glob(os.path.join(local_dir, "*_stats_history.csv"))
    if hist:
        try:
            dfh = pd.read_csv(hist[0])
            tcol = next(
                (
                    c
                    for c in ["Timestamp", "timestamp", "Time", "time"]
                    if c in dfh.columns
                ),
                None,
            )
            if tcol:
                s = dfh[tcol].dropna()
                if not s.empty:
                    if pd.api.types.is_integer_dtype(s) or pd.api.types.is_float_dtype(
                        s
                    ):
                        vals = pd.to_numeric(s, errors="coerce").dropna()
                        if not vals.empty:
                            vmax = float(vals.max())
                            if vmax > 1e12:
                                ts = pd.to_datetime(
                                    vals, unit="ns", errors="coerce", utc=True
                                ).dropna()
                            elif vmax > 1e10:
                                ts = pd.to_datetime(
                                    vals, unit="ms", errors="coerce", utc=True
                                ).dropna()
                            elif vmax > 1e9:
                                ts = pd.to_datetime(
                                    vals, unit="s", errors="coerce", utc=True
                                ).dropna()
                            else:
                                if end_guess is not None and vmax > 0:
                                    end_dt = end_guess
                                    start_dt = end_dt - timedelta(seconds=vmax)
                                    return start_dt, end_dt
                                ts = None
                            if ts is not None and not ts.empty:
                                return (
                                    ts.iloc[0].to_pydatetime(),
                                    ts.iloc[-1].to_pydatetime(),
                                )
                    else:
                        ts = pd.to_datetime(s, errors="coerce", utc=True).dropna()
                        if not ts.empty:
                            return (
                                ts.iloc[0].to_pydatetime(),
                                ts.iloc[-1].to_pydatetime(),
                            )
        except Exception:
            pass
    if end_guess is not None:
        dur = _parse_label_duration_seconds(item.get("label", ""))
        if dur and dur > 0:
            return end_guess - timedelta(seconds=dur), end_guess
    return None, None


def summarize_with_retry(
    lb_id: str, metric: str, s_iso: str, e_iso: str, res="1m", agg="sum"
):
    delay = 0.4
    for attempt in range(5):
        try:
            return get_lb_metric_df(
                lb_id, metric, s_iso, e_iso, resolution=res, agg=agg
            )
        except oci.exceptions.ServiceError as e:
            if getattr(e, "status", None) == 429 and attempt < 4:
                time.sleep(delay)
                delay *= 1.7
                continue
            raise
    return pd.DataFrame(columns=["timestamp", "value"])


metrics_root = os.path.join(OUTPUT_DIR, f"oci_metrics_{TS_UTC}")
os.makedirs(metrics_root, exist_ok=True)

index_summary = {
    "lb_id": LB_ID,
    "lb_ips": LB_IPS,
    "generated_at_utc": _iso_utc(datetime.now(timezone.utc)),
    "runs": [],
}
any_written = False

for item in SUITE_RESULTS or []:
    if not item:
        continue
    label = item.get("label", "(no-label)")
    s_dt, e_dt = item.get("start_ts"), item.get("end_ts")
    if s_dt is None or e_dt is None:
        s_dt, e_dt = _derive_window_from_artifacts(item)

    run_entry = {"label": label, "status": "ok", "paths": {}}
    if s_dt is None or e_dt is None:
        msg = f"[{label}] no usable time window; skipping metrics"
        print(msg)
        run_entry["status"] = "no-window"
        run_entry["message"] = msg
        index_summary["runs"].append(run_entry)
        continue

    start = s_dt - timedelta(seconds=5)
    end = e_dt + timedelta(seconds=5)
    s_iso, e_iso = _iso_utc(start), _iso_utc(end)
    window_seconds = max(1, int((end - start).total_seconds()))

    print(f"\n[label={label}] time_window: {s_iso} .. {e_iso}")

    run_dir = os.path.join(metrics_root, label)
    os.makedirs(run_dir, exist_ok=True)
    window_path = os.path.join(run_dir, "window.json")
    with open(window_path, "w") as f:
        json.dump({"start_iso": s_iso, "end_iso": e_iso}, f, indent=2)
    run_entry["paths"]["window_json"] = window_path

    per_metric = {}
    df_bytes_sent = None
    bytes_sent_metric_name = None

    for key in ["CPS", "Active", "Closed", "BytesIn", "BytesOut"]:
        metric = pick_metric(key)
        if not metric:
            print(f"  {key}: no alias available in namespace {LB_METRICS_NAMESPACE}")
            per_metric[key] = {
                "metric": None,
                "points": 0,
                "total": 0,
                "note": "alias-missing",
            }
            continue
        try:
            df = summarize_with_retry(LB_ID, metric, s_iso, e_iso, "1m", "sum")
            total = df["value"].sum() if not df.empty else 0
            points = len(df) if not df.empty else 0
            print(f"  {metric}: points={points} total={total}")
            csv_path = os.path.join(run_dir, f"{metric}.csv")
            df.to_csv(csv_path, index=False)
            per_metric[key] = {
                "metric": metric,
                "points": int(points),
                "total": float(total),
                "csv": csv_path,
            }
            if metric in ("BytesSent", "BytesOut"):
                df_bytes_sent = df.copy()
                bytes_sent_metric_name = metric
        except Exception as e:
            print(f"  {metric}: error {e}")
            per_metric[key] = {"metric": metric, "error": str(e)}
        time.sleep(0.25)

    throughput = None
    if df_bytes_sent is not None and not df_bytes_sent.empty:
        df_bytes_sent["gbps"] = (df_bytes_sent["value"] * 8.0) / 60.0 / 1e9
        total_bytes = float(df_bytes_sent["value"].sum())
        avg_gbps = (total_bytes * 8.0) / window_seconds / 1e9
        peak_gbps = (
            float(df_bytes_sent["gbps"].max())
            if not df_bytes_sent["gbps"].empty
            else 0.0
        )
        tput_csv = os.path.join(run_dir, "ThroughputGbps.csv")
        df_bytes_sent[["timestamp", "gbps"]].to_csv(tput_csv, index=False)
        throughput = {
            "metric_source": bytes_sent_metric_name,
            "avg_gbps": round(avg_gbps, 6),
            "peak_gbps": round(peak_gbps, 6),
            "series_csv": tput_csv,
        }
        print(
            f"  Throughput (from {bytes_sent_metric_name}): avg={avg_gbps:.6f} Gbps, peak={peak_gbps:.6f} Gbps"
        )
    else:
        print("  Throughput: BytesSent/BytesOut series not available; skipping Gbps.")

    summary_path = os.path.join(run_dir, "summary.json")
    with open(summary_path, "w") as f:
        json.dump(
            {
                "label": label,
                "start_iso": s_iso,
                "end_iso": e_iso,
                "metrics": per_metric,
                "throughput": throughput,
            },
            f,
            indent=2,
        )
    run_entry["paths"]["summary_json"] = summary_path
    run_entry["status"] = "ok"
    any_written = True

    print(f"  Saved run artifacts to: {run_dir}")
    index_summary["runs"].append(run_entry)

index_path = os.path.join(metrics_root, "index.json")
with open(index_path, "w") as f:
    json.dump(index_summary, f, indent=2)

if any_written:
    print(f"\nCell 18 complete: Wrote OCI metrics artifacts to: {metrics_root}")
    print(f"Index: {index_path}")
else:
    print("\nCell 18 complete: No metrics written (no usable windows).")
print(
    "NEXT: Run Cell 19 for concise local CSV/HTML summaries; Cell 20 to teardown (optional)."
)

In [ ]:
# Cell 19 — Objective: Local artifact summary, optional plots

import glob, os, pandas as pd


def summarize_locust_csv(local_dir):
    paths = glob.glob(os.path.join(local_dir, "*_stats.csv"))
    if not paths:
        print(local_dir, "No *_stats.csv found")
        return
    for p in paths:
        try:
            df = pd.read_csv(p)
            if "Name" in df.columns:
                row = (
                    df[df["Name"].str.lower().eq("aggregated")]
                    if any(df["Name"].str.lower().eq("aggregated"))
                    else df[df["Name"].str.lower().eq("total")]
                )
                overall = row if not row.empty else df.tail(1)
            else:
                overall = df.tail(1)
            print("\nFile:", os.path.basename(p))
            print(overall.to_string(index=False))
        except Exception as e:
            print("Could not parse", p, e)


for item in SUITE_RESULTS or []:
    if not item:
        continue
    print("\n===", item["label"], "===")
    summarize_locust_csv(item["local_dir"])
    htmls = glob.glob(os.path.join(item["local_dir"], "*.html"))
    if htmls:
        print("Report HTML(s):")
        for h in htmls:
            print(" -", h)

print("Cell 19 complete: Local CSV/HTML summaries printed.")
print("NEXT: If you want to tear everything down, run Cell 20.")

In [ ]:
# Cell 20 — Objective: Teardown (guarded)

TEARDOWN_CONFIRM = True  # Set True to allow destroy

if TEARDOWN_CONFIRM:
    !terraform destroy -auto-approve -var-file=terraform.tfvars
    print("Cell 20 complete: All Terraform resources destroyed.")
else:
    print("Teardown guard is False. Set TEARDOWN_CONFIRM=True to destroy resources.")
    print("Cell 20 complete: No teardown executed.")
